In [1]:
from __future__ import annotations

import json
import math

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from IPython.display import display


# =====================================================================
# Project paths
# =====================================================================

THESIS_DIR = Path(
    "/home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS"
)

CHAPTER4_ROOT = THESIS_DIR / "outputs" / "chapter4"

SINGLE_RUN_ROOT = (
    CHAPTER4_ROOT
    / "hybrid_single_ended"
    / "1751690_20260718_001736"
)

DOUBLE_RUN_ROOT = (
    CHAPTER4_ROOT
    / "hybrid_double_ended"
    / "1751934_20260718_115538"
)

SINGLE_POSTHOC_ROOT = SINGLE_RUN_ROOT / "posthoc_metrics"
DOUBLE_POSTHOC_ROOT = DOUBLE_RUN_ROOT / "posthoc_metrics"

FINAL_DIR = CHAPTER4_ROOT / "posthoc_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# =====================================================================
# Mandatory inputs
# =====================================================================

INTEGRITY_JSON_PATH = (
    FINAL_DIR
    / "posthoc_integrity_audit.json"
)

FINAL_MANIFEST_PATH = (
    FINAL_DIR
    / "experiment_artifact_manifest_final.json"
)

SINGLE_WINDOW_PATH = (
    SINGLE_POSTHOC_ROOT
    / "single_ended_oof_window_predictions.parquet"
)

DOUBLE_WINDOW_PATH = (
    DOUBLE_POSTHOC_ROOT
    / "double_ended_oof_window_predictions.parquet"
)


# =====================================================================
# Outputs
# =====================================================================

UNIFIED_WINDOW_PATH = (
    FINAL_DIR
    / "posthoc_unified_window_predictions.parquet"
)

UNIFIED_EVENT_PATH = (
    FINAL_DIR
    / "posthoc_unified_event_predictions.parquet"
)

UNIFIED_AUDIT_PATH = (
    FINAL_DIR
    / "02_unified_prediction_table_audit.csv"
)

PRIOR_IDENTITY_AUDIT_PATH = (
    FINAL_DIR
    / "02_correction_learning_prior_identity_audit.csv"
)

MAPPING_RECORD_PATH = (
    FINAL_DIR
    / "posthoc_prediction_mapping_record.json"
)

COMPLETION_PATH = (
    FINAL_DIR
    / "02_unified_predictions_completion_summary.json"
)


# =====================================================================
# Numerical tolerances
# =====================================================================

NUMERIC_TOLERANCE = 1e-8
PRIOR_TOLERANCE = 1e-6


for required_path in [
    INTEGRITY_JSON_PATH,
    FINAL_MANIFEST_PATH,
    SINGLE_WINDOW_PATH,
    DOUBLE_WINDOW_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


print("Integrity audit:", INTEGRITY_JSON_PATH)
print("Unified window output:", UNIFIED_WINDOW_PATH)
print("Unified event output:", UNIFIED_EVENT_PATH)

Integrity audit: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_integrity_audit.json
Unified window output: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_window_predictions.parquet
Unified event output: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_event_predictions.parquet


In [2]:
with INTEGRITY_JSON_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    integrity_payload = json.load(file)


integrity_status = integrity_payload.get(
    "overall_status"
)


print("Stored integrity status:", integrity_status)
print(
    "Mandatory failures:",
    integrity_payload.get(
        "mandatory_failure_count"
    ),
)


if integrity_status != "PASS":
    raise RuntimeError(
        "The mandatory integrity audit did not pass. "
        "Unified prediction tables cannot be built."
    )


with FINAL_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    final_manifest = json.load(file)


print(
    "Manifest version:",
    final_manifest.get("manifest_version"),
)

print(
    "Experiments in manifest:",
    len(final_manifest.get("experiments", {})),
)

print("\nIntegrity gate passed.")

Stored integrity status: PASS
Mandatory failures: 0
Manifest version: chapter4_posthoc_final_v2
Experiments in manifest: 8

Integrity gate passed.


In [3]:
EXPERIMENT_ORDER = [
    "C90-1E",
    "L90-1E",
    "C110-1E",
    "L110-1E",
    "C90-2E",
    "L90-2E",
    "C110-2E",
    "L110-2E",
]


EXPERIMENTS: dict[str, dict[str, Any]] = {
    "C90-1E": {
        "topology": "90kv",
        "prior_view": "1E",
        "model_family": "correction",
        "source_path": SINGLE_WINDOW_PATH,
        "expected_rows": 9022,
        "expected_events": 9022,
        "expected_windows": 1,
    },
    "L90-1E": {
        "topology": "90kv",
        "prior_view": "1E",
        "model_family": "combination",
        "source_path": SINGLE_WINDOW_PATH,
        "expected_rows": 9022,
        "expected_events": 9022,
        "expected_windows": 1,
    },
    "C110-1E": {
        "topology": "110kv",
        "prior_view": "1E",
        "model_family": "correction",
        "source_path": SINGLE_WINDOW_PATH,
        "expected_rows": 3648,
        "expected_events": 912,
        "expected_windows": 4,
    },
    "L110-1E": {
        "topology": "110kv",
        "prior_view": "1E",
        "model_family": "combination",
        "source_path": SINGLE_WINDOW_PATH,
        "expected_rows": 3648,
        "expected_events": 912,
        "expected_windows": 4,
    },
    "C90-2E": {
        "topology": "90kv",
        "prior_view": "2E",
        "model_family": "correction",
        "source_path": DOUBLE_WINDOW_PATH,
        "expected_rows": 9022,
        "expected_events": 9022,
        "expected_windows": 1,
    },
    "L90-2E": {
        "topology": "90kv",
        "prior_view": "2E",
        "model_family": "combination",
        "source_path": DOUBLE_WINDOW_PATH,
        "expected_rows": 9022,
        "expected_events": 9022,
        "expected_windows": 1,
    },
    "C110-2E": {
        "topology": "110kv",
        "prior_view": "2E",
        "model_family": "correction",
        "source_path": DOUBLE_WINDOW_PATH,
        "expected_rows": 3648,
        "expected_events": 912,
        "expected_windows": 4,
    },
    "L110-2E": {
        "topology": "110kv",
        "prior_view": "2E",
        "model_family": "combination",
        "source_path": DOUBLE_WINDOW_PATH,
        "expected_rows": 3648,
        "expected_events": 912,
        "expected_windows": 4,
    },
}


def normalize_identifier(value: Any) -> str | None:
    if pd.isna(value):
        return None

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        if np.isfinite(value) and float(value).is_integer():
            return str(int(value))

    return str(value).strip()


def optional_series(
    frame: pd.DataFrame,
    column: str,
    default: Any = np.nan,
) -> pd.Series:
    if column in frame.columns:
        return frame[column].copy()

    return pd.Series(
        default,
        index=frame.index,
    )


def broad_fault_family(case_value: Any) -> str:
    if pd.isna(case_value):
        return "unknown"

    token = (
        str(case_value)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    while "__" in token:
        token = token.replace("__", "_")

    if (
        token.startswith("3ph")
        or token.startswith("three_phase")
        or token.startswith("threephase")
    ):
        return "three_phase"

    if token.startswith("llg"):
        return "line_to_line_ground"

    if token.startswith("ll"):
        return "line_to_line"

    if (
        token.startswith("slg")
        or token.startswith("lg")
    ):
        return "single_line_ground"

    return "other"


def location_bin_10pp(value: Any) -> str | None:
    if pd.isna(value):
        return None

    numeric = float(value)

    if not np.isfinite(numeric):
        return None

    clipped = min(max(numeric, 0.0), 100.0)

    lower = min(
        int(math.floor(clipped / 10.0)) * 10,
        90,
    )

    upper = lower + 10

    return f"{lower:02d}-{upper:02d}"


def numeric_std_zero_ddof(
    values: pd.Series,
) -> float:
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    numeric = numeric[
        np.isfinite(numeric)
    ].to_numpy(dtype=float)

    if len(numeric) == 0:
        return np.nan

    return float(
        np.std(
            numeric,
            ddof=0,
        )
    )


def numeric_range(
    values: pd.Series,
) -> float:
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    numeric = numeric[
        np.isfinite(numeric)
    ]

    if len(numeric) == 0:
        return np.nan

    return float(
        numeric.max()
        - numeric.min()
    )


def maximum_difference(
    left: pd.Series,
    right: pd.Series,
) -> float:
    left_numeric = pd.to_numeric(
        left,
        errors="coerce",
    )

    right_numeric = pd.to_numeric(
        right,
        errors="coerce",
    )

    valid = (
        np.isfinite(left_numeric)
        & np.isfinite(right_numeric)
    )

    if not valid.any():
        return np.inf

    return float(
        (
            left_numeric[valid]
            - right_numeric[valid]
        ).abs().max()
    )

In [4]:
single_window = pd.read_parquet(
    SINGLE_WINDOW_PATH
)

double_window = pd.read_parquet(
    DOUBLE_WINDOW_PATH
)


print("Single-ended source shape:", single_window.shape)
print("Double-ended source shape:", double_window.shape)


assert len(single_window) == 25_340
assert len(double_window) == 25_340


required_source_columns = {
    "experiment_id",
    "sample_id",
    "fold",
    "y_true_pp",
    "y_pred_pp",
    "prior_pp",
    "window_idx",
    "case",
    "y_fault_line",
    "event_type",
    "model_abs_error_pp",
    "prior_abs_error_pp",
}


for label, frame in {
    "single-ended": single_window,
    "double-ended": double_window,
}.items():
    missing = sorted(
        required_source_columns
        - set(frame.columns)
    )

    if missing:
        raise KeyError(
            f"{label} source is missing: {missing}"
        )


print("\nSource experiment counts:")

display(
    pd.concat(
        [
            single_window[
                "experiment_id"
            ].value_counts().rename(
                "single_rows"
            ),
            double_window[
                "experiment_id"
            ].value_counts().rename(
                "double_rows"
            ),
        ],
        axis=1,
    ).fillna(0).astype(int)
)

Single-ended source shape: (25340, 23)
Double-ended source shape: (25340, 15)

Source experiment counts:


,single_rows,double_rows
experiment_id,,
C90-1E,9022,0
L90-1E,9022,0
C110-1E,3648,0
L110-1E,3648,0
C90-2E,0,9022
L90-2E,0,9022
C110-2E,0,3648
L110-2E,0,3648


In [5]:
window_frames = []
window_audit_rows = []


for experiment_id in EXPERIMENT_ORDER:
    config = EXPERIMENTS[
        experiment_id
    ]

    if config["prior_view"] == "1E":
        source = single_window
    else:
        source = double_window

    source_slice = source.loc[
        source["experiment_id"]
        == experiment_id
    ].copy().reset_index(drop=True)

    unified = pd.DataFrame(
        {
            "experiment_id": experiment_id,
            "topology": config["topology"],
            "prior_view": config["prior_view"],
            "model_family": config[
                "model_family"
            ],

            "sample_id": source_slice[
                "sample_id"
            ].map(normalize_identifier),

            "fold": pd.to_numeric(
                source_slice["fold"],
                errors="coerce",
            ).astype("Int64"),

            "window_idx": pd.to_numeric(
                source_slice["window_idx"],
                errors="coerce",
            ).astype("Int64"),

            "y_true_pct": pd.to_numeric(
                source_slice["y_true_pp"],
                errors="coerce",
            ),

            "y_pred_pct": pd.to_numeric(
                source_slice["y_pred_pp"],
                errors="coerce",
            ),

            "y_prior_pct": pd.to_numeric(
                source_slice["prior_pp"],
                errors="coerce",
            ),

            "case": source_slice[
                "case"
            ].astype("string"),

            "y_fault_line": source_slice[
                "y_fault_line"
            ].astype("string"),

            "event_type": source_slice[
                "event_type"
            ].astype("string"),

            "status": optional_series(
                source_slice,
                "status",
                pd.NA,
            ).astype("string"),

            "case_idx": pd.to_numeric(
                optional_series(
                    source_slice,
                    "case_idx",
                ),
                errors="coerce",
            ).astype("Int64"),

            "idx_test": pd.to_numeric(
                optional_series(
                    source_slice,
                    "idx_test",
                ),
                errors="coerce",
            ).astype("Int64"),

            "alpha": pd.to_numeric(
                optional_series(
                    source_slice,
                    "alpha",
                ),
                errors="coerce",
            ),

            "residual": pd.to_numeric(
                optional_series(
                    source_slice,
                    "residual",
                ),
                errors="coerce",
            ),

            "d_prior_internal": pd.to_numeric(
                optional_series(
                    source_slice,
                    "d_prior",
                ),
                errors="coerce",
            ),

            "source_prediction_file": (
                optional_series(
                    source_slice,
                    "prediction_file",
                    pd.NA,
                ).astype("string")
            ),

            "source_oof_path": str(
                config["source_path"]
            ),
        }
    )

    unified[
        "model_abs_error_pp"
    ] = (
        unified["y_pred_pct"]
        - unified["y_true_pct"]
    ).abs()

    unified[
        "prior_abs_error_pp"
    ] = (
        unified["y_prior_pct"]
        - unified["y_true_pct"]
    ).abs()

    # Positive means the model improved over the prior.
    unified[
        "error_change_pp"
    ] = (
        unified["prior_abs_error_pp"]
        - unified["model_abs_error_pp"]
    )

    unified[
        "broad_fault_family"
    ] = unified["case"].map(
        broad_fault_family
    )

    unified[
        "location_pct"
    ] = unified["y_true_pct"]

    unified[
        "location_bin_10pp"
    ] = unified["location_pct"].map(
        location_bin_10pp
    )

    unified[
        "physical_event_key"
    ] = (
        unified["topology"].astype(str)
        + "::"
        + unified["sample_id"].astype(str)
    )

    unified[
        "cohort_event_key"
    ] = (
        unified["topology"].astype(str)
        + "::"
        + unified["prior_view"].astype(str)
        + "::"
        + unified["sample_id"].astype(str)
    )

    unified[
        "experiment_event_key"
    ] = (
        unified["experiment_id"].astype(str)
        + "::"
        + unified["sample_id"].astype(str)
    )

    row_count = len(unified)

    event_count = unified[
        "sample_id"
    ].nunique(dropna=False)

    rows_per_event = unified.groupby(
        "sample_id",
        dropna=False,
    ).size()

    duplicate_keys = int(
        unified.duplicated(
            subset=[
                "experiment_id",
                "sample_id",
                "window_idx",
            ],
            keep=False,
        ).sum()
    )

    finite_required = (
        np.isfinite(unified["y_true_pct"])
        & np.isfinite(unified["y_pred_pct"])
        & np.isfinite(unified["y_prior_pct"])
    )

    mapping_ok = bool(
        row_count == config["expected_rows"]
        and event_count
        == config["expected_events"]
        and (
            rows_per_event
            == config["expected_windows"]
        ).all()
        and duplicate_keys == 0
        and finite_required.all()
    )

    window_audit_rows.append(
        {
            "level": "window",
            "experiment_id": experiment_id,
            "rows": row_count,
            "expected_rows": config[
                "expected_rows"
            ],
            "events": event_count,
            "expected_events": config[
                "expected_events"
            ],
            "windows_per_event_min": int(
                rows_per_event.min()
            ),
            "windows_per_event_max": int(
                rows_per_event.max()
            ),
            "expected_windows_per_event": (
                config["expected_windows"]
            ),
            "duplicate_keys": duplicate_keys,
            "nonfinite_required_rows": int(
                (~finite_required).sum()
            ),
            "mapping_ok": mapping_ok,
        }
    )

    window_frames.append(unified)


unified_window = pd.concat(
    window_frames,
    ignore_index=True,
)


experiment_order_map = {
    experiment_id: index
    for index, experiment_id
    in enumerate(EXPERIMENT_ORDER)
}

unified_window["_experiment_order"] = (
    unified_window["experiment_id"].map(
        experiment_order_map
    )
)

unified_window = (
    unified_window
    .sort_values(
        [
            "_experiment_order",
            "sample_id",
            "window_idx",
        ],
        kind="stable",
    )
    .drop(columns="_experiment_order")
    .reset_index(drop=True)
)


window_audit = pd.DataFrame(
    window_audit_rows
)

display(window_audit)


if not window_audit["mapping_ok"].all():
    raise RuntimeError(
        "The unified window table failed a mandatory "
        "row, event, duplicate, or numeric check."
    )


unified_window.to_parquet(
    UNIFIED_WINDOW_PATH,
    index=False,
)

print(
    "Saved unified window table:",
    UNIFIED_WINDOW_PATH,
)

print(
    "Unified window rows:",
    f"{len(unified_window):,}",
)

,level,experiment_id,rows,expected_rows,events,expected_events,windows_per_event_min,windows_per_event_max,expected_windows_per_event,duplicate_keys,nonfinite_required_rows,mapping_ok
0,window,C90-1E,9022,9022,9022,9022,1,1,1,0,0,True
1,window,L90-1E,9022,9022,9022,9022,1,1,1,0,0,True
2,window,C110-1E,3648,3648,912,912,4,4,4,0,0,True
3,window,L110-1E,3648,3648,912,912,4,4,4,0,0,True
4,window,C90-2E,9022,9022,9022,9022,1,1,1,0,0,True
5,window,L90-2E,9022,9022,9022,9022,1,1,1,0,0,True
6,window,C110-2E,3648,3648,912,912,4,4,4,0,0,True
7,window,L110-2E,3648,3648,912,912,4,4,4,0,0,True


Saved unified window table: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_window_predictions.parquet
Unified window rows: 50,680


In [6]:
event_frames = []
event_audit_rows = []


for experiment_id in EXPERIMENT_ORDER:
    config = EXPERIMENTS[
        experiment_id
    ]

    frame = unified_window.loc[
        unified_window["experiment_id"]
        == experiment_id
    ].copy()

    grouped = frame.groupby(
        "sample_id",
        sort=False,
        dropna=False,
    )

    event = grouped.agg(
        experiment_id=(
            "experiment_id",
            "first",
        ),
        topology=(
            "topology",
            "first",
        ),
        prior_view=(
            "prior_view",
            "first",
        ),
        model_family=(
            "model_family",
            "first",
        ),
        fold=(
            "fold",
            "first",
        ),
        y_true_pct=(
            "y_true_pct",
            "first",
        ),
        y_pred_pct=(
            "y_pred_pct",
            "mean",
        ),
        y_pred_median_pct=(
            "y_pred_pct",
            "median",
        ),
        y_pred_min_pct=(
            "y_pred_pct",
            "min",
        ),
        y_pred_max_pct=(
            "y_pred_pct",
            "max",
        ),
        y_pred_std_pct=(
            "y_pred_pct",
            numeric_std_zero_ddof,
        ),
        y_pred_range_pct=(
            "y_pred_pct",
            numeric_range,
        ),
        y_prior_pct=(
            "y_prior_pct",
            "mean",
        ),
        y_prior_min_pct=(
            "y_prior_pct",
            "min",
        ),
        y_prior_max_pct=(
            "y_prior_pct",
            "max",
        ),
        case=(
            "case",
            "first",
        ),
        broad_fault_family=(
            "broad_fault_family",
            "first",
        ),
        y_fault_line=(
            "y_fault_line",
            "first",
        ),
        event_type=(
            "event_type",
            "first",
        ),
        status=(
            "status",
            "first",
        ),
        case_idx=(
            "case_idx",
            "first",
        ),
        window_rows=(
            "sample_id",
            "size",
        ),
        window_idx_min=(
            "window_idx",
            "min",
        ),
        window_idx_max=(
            "window_idx",
            "max",
        ),
        window_model_abs_error_mean_pp=(
            "model_abs_error_pp",
            "mean",
        ),
        window_model_abs_error_min_pp=(
            "model_abs_error_pp",
            "min",
        ),
        window_model_abs_error_max_pp=(
            "model_abs_error_pp",
            "max",
        ),
        window_prior_abs_error_mean_pp=(
            "prior_abs_error_pp",
            "mean",
        ),
        alpha_mean=(
            "alpha",
            "mean",
        ),
        alpha_std=(
            "alpha",
            numeric_std_zero_ddof,
        ),
        alpha_min=(
            "alpha",
            "min",
        ),
        alpha_max=(
            "alpha",
            "max",
        ),
        residual_mean=(
            "residual",
            "mean",
        ),
        residual_abs_mean=(
            "residual",
            lambda values: pd.to_numeric(
                values,
                errors="coerce",
            ).abs().mean(),
        ),
        residual_std=(
            "residual",
            numeric_std_zero_ddof,
        ),
        target_min_pct=(
            "y_true_pct",
            "min",
        ),
        target_max_pct=(
            "y_true_pct",
            "max",
        ),
        fold_count=(
            "fold",
            "nunique",
        ),
        case_count=(
            "case",
            "nunique",
        ),
        line_count=(
            "y_fault_line",
            "nunique",
        ),
        event_type_count=(
            "event_type",
            "nunique",
        ),
    ).reset_index()

    event[
        "target_spread_pp"
    ] = (
        event["target_max_pct"]
        - event["target_min_pct"]
    )

    event[
        "prior_window_spread_pp"
    ] = (
        event["y_prior_max_pct"]
        - event["y_prior_min_pct"]
    )

    event[
        "model_abs_error_pp"
    ] = (
        event["y_pred_pct"]
        - event["y_true_pct"]
    ).abs()

    event[
        "prior_abs_error_pp"
    ] = (
        event["y_prior_pct"]
        - event["y_true_pct"]
    ).abs()

    # Positive means model improvement.
    event[
        "error_change_pp"
    ] = (
        event["prior_abs_error_pp"]
        - event["model_abs_error_pp"]
    )

    # Positive means averaging the windows reduced the error.
    event[
        "event_averaging_gain_pp"
    ] = (
        event[
            "window_model_abs_error_mean_pp"
        ]
        - event["model_abs_error_pp"]
    )

    event[
        "location_pct"
    ] = event["y_true_pct"]

    event[
        "location_bin_10pp"
    ] = event["location_pct"].map(
        location_bin_10pp
    )

    event[
        "physical_event_key"
    ] = (
        event["topology"].astype(str)
        + "::"
        + event["sample_id"].astype(str)
    )

    event[
        "cohort_event_key"
    ] = (
        event["topology"].astype(str)
        + "::"
        + event["prior_view"].astype(str)
        + "::"
        + event["sample_id"].astype(str)
    )

    event[
        "experiment_event_key"
    ] = (
        event["experiment_id"].astype(str)
        + "::"
        + event["sample_id"].astype(str)
    )

    event[
        "aggregation_method"
    ] = np.where(
        event["topology"] == "110kv",
        "arithmetic_mean_of_four_windows",
        "single_retained_window",
    )

    expected_events = config[
        "expected_events"
    ]

    row_count_ok = (
        len(event) == expected_events
    )

    window_rows_ok = bool(
        (
            event["window_rows"]
            == config["expected_windows"]
        ).all()
    )

    target_ok = bool(
        (
            event["target_spread_pp"]
            <= NUMERIC_TOLERANCE
        ).all()
    )

    fold_ok = bool(
        (event["fold_count"] == 1).all()
    )

    metadata_ok = bool(
        (event["case_count"] == 1).all()
        and (event["line_count"] == 1).all()
    )

    finite_required = (
        np.isfinite(event["y_true_pct"])
        & np.isfinite(event["y_pred_pct"])
        & np.isfinite(event["y_prior_pct"])
        & np.isfinite(
            event["model_abs_error_pp"]
        )
        & np.isfinite(
            event["prior_abs_error_pp"]
        )
    )

    event_ok = bool(
        row_count_ok
        and window_rows_ok
        and target_ok
        and fold_ok
        and metadata_ok
        and finite_required.all()
    )

    event_audit_rows.append(
        {
            "level": "event",
            "experiment_id": experiment_id,
            "events": len(event),
            "expected_events": expected_events,
            "window_rows_min": int(
                event["window_rows"].min()
            ),
            "window_rows_max": int(
                event["window_rows"].max()
            ),
            "expected_windows_per_event": (
                config["expected_windows"]
            ),
            "events_with_target_spread": int(
                (
                    event["target_spread_pp"]
                    > NUMERIC_TOLERANCE
                ).sum()
            ),
            "events_in_multiple_folds": int(
                (event["fold_count"] > 1).sum()
            ),
            "events_with_case_conflict": int(
                (event["case_count"] > 1).sum()
            ),
            "events_with_line_conflict": int(
                (event["line_count"] > 1).sum()
            ),
            "nonfinite_required_events": int(
                (~finite_required).sum()
            ),
            "aggregation_ok": event_ok,
        }
    )

    event_frames.append(event)


unified_event = pd.concat(
    event_frames,
    ignore_index=True,
)


unified_event["_experiment_order"] = (
    unified_event["experiment_id"].map(
        experiment_order_map
    )
)

unified_event = (
    unified_event
    .sort_values(
        [
            "_experiment_order",
            "sample_id",
        ],
        kind="stable",
    )
    .drop(columns="_experiment_order")
    .reset_index(drop=True)
)


event_audit = pd.DataFrame(
    event_audit_rows
)

display(event_audit)


if not event_audit["aggregation_ok"].all():
    raise RuntimeError(
        "The unified event table failed a mandatory "
        "aggregation or cohort check."
    )


unified_event.to_parquet(
    UNIFIED_EVENT_PATH,
    index=False,
)

print(
    "Saved unified event table:",
    UNIFIED_EVENT_PATH,
)

print(
    "Unified event rows:",
    f"{len(unified_event):,}",
)

,level,experiment_id,events,expected_events,window_rows_min,window_rows_max,expected_windows_per_event,events_with_target_spread,events_in_multiple_folds,events_with_case_conflict,events_with_line_conflict,nonfinite_required_events,aggregation_ok
0,event,C90-1E,9022,9022,1,1,1,0,0,0,0,0,True
1,event,L90-1E,9022,9022,1,1,1,0,0,0,0,0,True
2,event,C110-1E,912,912,4,4,4,0,0,0,0,0,True
3,event,L110-1E,912,912,4,4,4,0,0,0,0,0,True
4,event,C90-2E,9022,9022,1,1,1,0,0,0,0,0,True
5,event,L90-2E,9022,9022,1,1,1,0,0,0,0,0,True
6,event,C110-2E,912,912,4,4,4,0,0,0,0,0,True
7,event,L110-2E,912,912,4,4,4,0,0,0,0,0,True


Saved unified event table: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_event_predictions.parquet
Unified event rows: 39,736


In [7]:
event_frames = []
event_audit_rows = []


for experiment_id in EXPERIMENT_ORDER:
    config = EXPERIMENTS[
        experiment_id
    ]

    frame = unified_window.loc[
        unified_window["experiment_id"]
        == experiment_id
    ].copy()

    grouped = frame.groupby(
        "sample_id",
        sort=False,
        dropna=False,
    )

    event = grouped.agg(
        experiment_id=(
            "experiment_id",
            "first",
        ),
        topology=(
            "topology",
            "first",
        ),
        prior_view=(
            "prior_view",
            "first",
        ),
        model_family=(
            "model_family",
            "first",
        ),
        fold=(
            "fold",
            "first",
        ),
        y_true_pct=(
            "y_true_pct",
            "first",
        ),
        y_pred_pct=(
            "y_pred_pct",
            "mean",
        ),
        y_pred_median_pct=(
            "y_pred_pct",
            "median",
        ),
        y_pred_min_pct=(
            "y_pred_pct",
            "min",
        ),
        y_pred_max_pct=(
            "y_pred_pct",
            "max",
        ),
        y_pred_std_pct=(
            "y_pred_pct",
            numeric_std_zero_ddof,
        ),
        y_pred_range_pct=(
            "y_pred_pct",
            numeric_range,
        ),
        y_prior_pct=(
            "y_prior_pct",
            "mean",
        ),
        y_prior_min_pct=(
            "y_prior_pct",
            "min",
        ),
        y_prior_max_pct=(
            "y_prior_pct",
            "max",
        ),
        case=(
            "case",
            "first",
        ),
        broad_fault_family=(
            "broad_fault_family",
            "first",
        ),
        y_fault_line=(
            "y_fault_line",
            "first",
        ),
        event_type=(
            "event_type",
            "first",
        ),
        status=(
            "status",
            "first",
        ),
        case_idx=(
            "case_idx",
            "first",
        ),
        window_rows=(
            "sample_id",
            "size",
        ),
        window_idx_min=(
            "window_idx",
            "min",
        ),
        window_idx_max=(
            "window_idx",
            "max",
        ),
        window_model_abs_error_mean_pp=(
            "model_abs_error_pp",
            "mean",
        ),
        window_model_abs_error_min_pp=(
            "model_abs_error_pp",
            "min",
        ),
        window_model_abs_error_max_pp=(
            "model_abs_error_pp",
            "max",
        ),
        window_prior_abs_error_mean_pp=(
            "prior_abs_error_pp",
            "mean",
        ),
        alpha_mean=(
            "alpha",
            "mean",
        ),
        alpha_std=(
            "alpha",
            numeric_std_zero_ddof,
        ),
        alpha_min=(
            "alpha",
            "min",
        ),
        alpha_max=(
            "alpha",
            "max",
        ),
        residual_mean=(
            "residual",
            "mean",
        ),
        residual_abs_mean=(
            "residual",
            lambda values: pd.to_numeric(
                values,
                errors="coerce",
            ).abs().mean(),
        ),
        residual_std=(
            "residual",
            numeric_std_zero_ddof,
        ),
        target_min_pct=(
            "y_true_pct",
            "min",
        ),
        target_max_pct=(
            "y_true_pct",
            "max",
        ),
        fold_count=(
            "fold",
            "nunique",
        ),
        case_count=(
            "case",
            "nunique",
        ),
        line_count=(
            "y_fault_line",
            "nunique",
        ),
        event_type_count=(
            "event_type",
            "nunique",
        ),
    ).reset_index()

    event[
        "target_spread_pp"
    ] = (
        event["target_max_pct"]
        - event["target_min_pct"]
    )

    event[
        "prior_window_spread_pp"
    ] = (
        event["y_prior_max_pct"]
        - event["y_prior_min_pct"]
    )

    event[
        "model_abs_error_pp"
    ] = (
        event["y_pred_pct"]
        - event["y_true_pct"]
    ).abs()

    event[
        "prior_abs_error_pp"
    ] = (
        event["y_prior_pct"]
        - event["y_true_pct"]
    ).abs()

    # Positive means model improvement.
    event[
        "error_change_pp"
    ] = (
        event["prior_abs_error_pp"]
        - event["model_abs_error_pp"]
    )

    # Positive means averaging the windows reduced the error.
    event[
        "event_averaging_gain_pp"
    ] = (
        event[
            "window_model_abs_error_mean_pp"
        ]
        - event["model_abs_error_pp"]
    )

    event[
        "location_pct"
    ] = event["y_true_pct"]

    event[
        "location_bin_10pp"
    ] = event["location_pct"].map(
        location_bin_10pp
    )

    event[
        "physical_event_key"
    ] = (
        event["topology"].astype(str)
        + "::"
        + event["sample_id"].astype(str)
    )

    event[
        "cohort_event_key"
    ] = (
        event["topology"].astype(str)
        + "::"
        + event["prior_view"].astype(str)
        + "::"
        + event["sample_id"].astype(str)
    )

    event[
        "experiment_event_key"
    ] = (
        event["experiment_id"].astype(str)
        + "::"
        + event["sample_id"].astype(str)
    )

    event[
        "aggregation_method"
    ] = np.where(
        event["topology"] == "110kv",
        "arithmetic_mean_of_four_windows",
        "single_retained_window",
    )

    expected_events = config[
        "expected_events"
    ]

    row_count_ok = (
        len(event) == expected_events
    )

    window_rows_ok = bool(
        (
            event["window_rows"]
            == config["expected_windows"]
        ).all()
    )

    target_ok = bool(
        (
            event["target_spread_pp"]
            <= NUMERIC_TOLERANCE
        ).all()
    )

    fold_ok = bool(
        (event["fold_count"] == 1).all()
    )

    metadata_ok = bool(
        (event["case_count"] == 1).all()
        and (event["line_count"] == 1).all()
    )

    finite_required = (
        np.isfinite(event["y_true_pct"])
        & np.isfinite(event["y_pred_pct"])
        & np.isfinite(event["y_prior_pct"])
        & np.isfinite(
            event["model_abs_error_pp"]
        )
        & np.isfinite(
            event["prior_abs_error_pp"]
        )
    )

    event_ok = bool(
        row_count_ok
        and window_rows_ok
        and target_ok
        and fold_ok
        and metadata_ok
        and finite_required.all()
    )

    event_audit_rows.append(
        {
            "level": "event",
            "experiment_id": experiment_id,
            "events": len(event),
            "expected_events": expected_events,
            "window_rows_min": int(
                event["window_rows"].min()
            ),
            "window_rows_max": int(
                event["window_rows"].max()
            ),
            "expected_windows_per_event": (
                config["expected_windows"]
            ),
            "events_with_target_spread": int(
                (
                    event["target_spread_pp"]
                    > NUMERIC_TOLERANCE
                ).sum()
            ),
            "events_in_multiple_folds": int(
                (event["fold_count"] > 1).sum()
            ),
            "events_with_case_conflict": int(
                (event["case_count"] > 1).sum()
            ),
            "events_with_line_conflict": int(
                (event["line_count"] > 1).sum()
            ),
            "nonfinite_required_events": int(
                (~finite_required).sum()
            ),
            "aggregation_ok": event_ok,
        }
    )

    event_frames.append(event)


unified_event = pd.concat(
    event_frames,
    ignore_index=True,
)


unified_event["_experiment_order"] = (
    unified_event["experiment_id"].map(
        experiment_order_map
    )
)

unified_event = (
    unified_event
    .sort_values(
        [
            "_experiment_order",
            "sample_id",
        ],
        kind="stable",
    )
    .drop(columns="_experiment_order")
    .reset_index(drop=True)
)


event_audit = pd.DataFrame(
    event_audit_rows
)

display(event_audit)


if not event_audit["aggregation_ok"].all():
    raise RuntimeError(
        "The unified event table failed a mandatory "
        "aggregation or cohort check."
    )


unified_event.to_parquet(
    UNIFIED_EVENT_PATH,
    index=False,
)

print(
    "Saved unified event table:",
    UNIFIED_EVENT_PATH,
)

print(
    "Unified event rows:",
    f"{len(unified_event):,}",
)

,level,experiment_id,events,expected_events,window_rows_min,window_rows_max,expected_windows_per_event,events_with_target_spread,events_in_multiple_folds,events_with_case_conflict,events_with_line_conflict,nonfinite_required_events,aggregation_ok
0,event,C90-1E,9022,9022,1,1,1,0,0,0,0,0,True
1,event,L90-1E,9022,9022,1,1,1,0,0,0,0,0,True
2,event,C110-1E,912,912,4,4,4,0,0,0,0,0,True
3,event,L110-1E,912,912,4,4,4,0,0,0,0,0,True
4,event,C90-2E,9022,9022,1,1,1,0,0,0,0,0,True
5,event,L90-2E,9022,9022,1,1,1,0,0,0,0,0,True
6,event,C110-2E,912,912,4,4,4,0,0,0,0,0,True
7,event,L110-2E,912,912,4,4,4,0,0,0,0,0,True


Saved unified event table: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_event_predictions.parquet
Unified event rows: 39,736


In [8]:
aggregation_verification_rows = []


for experiment_id in EXPERIMENT_ORDER:
    config = EXPERIMENTS[
        experiment_id
    ]

    window = unified_window.loc[
        unified_window["experiment_id"]
        == experiment_id
    ].copy()

    event = unified_event.loc[
        unified_event["experiment_id"]
        == experiment_id
    ].copy()

    if config["topology"] == "90kv":
        comparison = window.merge(
            event[
                [
                    "sample_id",
                    "y_true_pct",
                    "y_pred_pct",
                    "y_prior_pct",
                ]
            ],
            on="sample_id",
            how="inner",
            suffixes=(
                "_window",
                "_event",
            ),
            validate="one_to_one",
        )

        prediction_max_difference = (
            maximum_difference(
                comparison[
                    "y_pred_pct_window"
                ],
                comparison[
                    "y_pred_pct_event"
                ],
            )
        )

        prior_max_difference = (
            maximum_difference(
                comparison[
                    "y_prior_pct_window"
                ],
                comparison[
                    "y_prior_pct_event"
                ],
            )
        )

        verification_ok = bool(
            len(comparison)
            == config["expected_events"]
            and prediction_max_difference
            <= NUMERIC_TOLERANCE
            and prior_max_difference
            <= PRIOR_TOLERANCE
        )

        aggregation_verification_rows.append(
            {
                "experiment_id": experiment_id,
                "topology": config["topology"],
                "verification": (
                    "window_equals_event"
                ),
                "matched_events": len(
                    comparison
                ),
                "prediction_max_difference": (
                    prediction_max_difference
                ),
                "prior_max_difference": (
                    prior_max_difference
                ),
                "verification_ok": (
                    verification_ok
                ),
            }
        )

    else:
        reconstructed = (
            window.groupby(
                "sample_id",
                dropna=False,
            )
            .agg(
                reconstructed_prediction=(
                    "y_pred_pct",
                    "mean",
                ),
                reconstructed_prior=(
                    "y_prior_pct",
                    "mean",
                ),
            )
            .reset_index()
        )

        comparison = reconstructed.merge(
            event[
                [
                    "sample_id",
                    "y_pred_pct",
                    "y_prior_pct",
                ]
            ],
            on="sample_id",
            how="inner",
            validate="one_to_one",
        )

        prediction_max_difference = (
            maximum_difference(
                comparison[
                    "reconstructed_prediction"
                ],
                comparison["y_pred_pct"],
            )
        )

        prior_max_difference = (
            maximum_difference(
                comparison[
                    "reconstructed_prior"
                ],
                comparison["y_prior_pct"],
            )
        )

        verification_ok = bool(
            len(comparison)
            == config["expected_events"]
            and prediction_max_difference
            <= NUMERIC_TOLERANCE
            and prior_max_difference
            <= PRIOR_TOLERANCE
        )

        aggregation_verification_rows.append(
            {
                "experiment_id": experiment_id,
                "topology": config["topology"],
                "verification": (
                    "event_equals_four_window_mean"
                ),
                "matched_events": len(
                    comparison
                ),
                "prediction_max_difference": (
                    prediction_max_difference
                ),
                "prior_max_difference": (
                    prior_max_difference
                ),
                "verification_ok": (
                    verification_ok
                ),
            }
        )


aggregation_verification = pd.DataFrame(
    aggregation_verification_rows
)

display(aggregation_verification)


if not aggregation_verification[
    "verification_ok"
].all():
    raise RuntimeError(
        "Event aggregation verification failed."
    )

,experiment_id,topology,verification,matched_events,prediction_max_difference,prior_max_difference,verification_ok
0,C90-1E,90kv,window_equals_event,9022,0.0,0.0,True
1,L90-1E,90kv,window_equals_event,9022,0.0,0.0,True
2,C110-1E,110kv,event_equals_four_window_mean,912,0.0,0.0,True
3,L110-1E,110kv,event_equals_four_window_mean,912,0.0,0.0,True
4,C90-2E,90kv,window_equals_event,9022,0.0,0.0,True
5,L90-2E,90kv,window_equals_event,9022,0.0,0.0,True
6,C110-2E,110kv,event_equals_four_window_mean,912,0.0,0.0,True
7,L110-2E,110kv,event_equals_four_window_mean,912,0.0,0.0,True


In [10]:
PAIR_DEFINITIONS = [
    {
        "topology": "90kv",
        "prior_view": "1E",
        "correction": "C90-1E",
        "combination": "L90-1E",
    },
    {
        "topology": "110kv",
        "prior_view": "1E",
        "correction": "C110-1E",
        "combination": "L110-1E",
    },
    {
        "topology": "90kv",
        "prior_view": "2E",
        "correction": "C90-2E",
        "combination": "L90-2E",
    },
    {
        "topology": "110kv",
        "prior_view": "2E",
        "correction": "C110-2E",
        "combination": "L110-2E",
    },
]


prior_identity_rows = []


for pair in PAIR_DEFINITIONS:
    for level, source, keys in [
        (
            "window",
            unified_window,
            [
                "sample_id",
                "window_idx",
            ],
        ),
        (
            "event",
            unified_event,
            ["sample_id"],
        ),
    ]:
        correction = source.loc[
            source["experiment_id"]
            == pair["correction"],
            keys
            + [
                "fold",
                "y_true_pct",
                "y_prior_pct",
                "case",
                "y_fault_line",
            ],
        ].copy()

        combination = source.loc[
            source["experiment_id"]
            == pair["combination"],
            keys
            + [
                "fold",
                "y_true_pct",
                "y_prior_pct",
                "case",
                "y_fault_line",
            ],
        ].copy()

        merged = correction.merge(
            combination,
            on=keys,
            how="outer",
            suffixes=(
                "_correction",
                "_combination",
            ),
            indicator=True,
            validate="one_to_one",
        )

        matched = merged.loc[
            merged["_merge"] == "both"
        ].copy()

        missing_correction = int(
            (
                merged["_merge"]
                == "right_only"
            ).sum()
        )

        missing_combination = int(
            (
                merged["_merge"]
                == "left_only"
            ).sum()
        )

        target_max_difference = (
            maximum_difference(
                matched[
                    "y_true_pct_correction"
                ],
                matched[
                    "y_true_pct_combination"
                ],
            )
        )

        prior_max_difference = (
            maximum_difference(
                matched[
                    "y_prior_pct_correction"
                ],
                matched[
                    "y_prior_pct_combination"
                ],
            )
        )

        fold_equal_mask = (
            matched["fold_correction"]
            .astype("Int64")
            .eq(
                matched["fold_combination"]
                .astype("Int64")
            )
        )

        fold_mismatches = int(
            (~fold_equal_mask).sum()
        )

        case_mismatches = int(
            (
                matched[
                    "case_correction"
                ]
                .fillna("<NA>")
                .astype(str)
                != matched[
                    "case_combination"
                ]
                .fillna("<NA>")
                .astype(str)
            ).sum()
        )

        line_mismatches = int(
            (
                matched[
                    "y_fault_line_correction"
                ]
                .fillna("<NA>")
                .astype(str)
                != matched[
                    "y_fault_line_combination"
                ]
                .fillna("<NA>")
                .astype(str)
            ).sum()
        )

        shared_prior_verified = bool(
            missing_correction == 0
            and missing_combination == 0
            and target_max_difference
            <= NUMERIC_TOLERANCE
            and prior_max_difference
            <= PRIOR_TOLERANCE
            and fold_mismatches == 0
            and case_mismatches == 0
            and line_mismatches == 0
        )

        prior_identity_rows.append(
            {
                "topology": pair["topology"],
                "prior_view": pair["prior_view"],
                "level": level,
                "correction_experiment": (
                    pair["correction"]
                ),
                "combination_experiment": (
                    pair["combination"]
                ),
                "correction_rows": len(
                    correction
                ),
                "combination_rows": len(
                    combination
                ),
                "matched_rows": len(matched),
                "missing_from_correction": (
                    missing_correction
                ),
                "missing_from_combination": (
                    missing_combination
                ),
                "target_max_difference": (
                    target_max_difference
                ),
                "prior_max_difference": (
                    prior_max_difference
                ),
                "fold_mismatches": fold_mismatches,
                "case_mismatches": case_mismatches,
                "line_mismatches": line_mismatches,
                "shared_prior_verified": (
                    shared_prior_verified
                ),
            }
        )


prior_identity_audit = pd.DataFrame(
    prior_identity_rows
)

prior_identity_audit.to_csv(
    PRIOR_IDENTITY_AUDIT_PATH,
    index=False,
)

display(prior_identity_audit)

print("Saved:", PRIOR_IDENTITY_AUDIT_PATH)

,topology,prior_view,level,correction_experiment,combination_experiment,correction_rows,combination_rows,matched_rows,missing_from_correction,missing_from_combination,target_max_difference,prior_max_difference,fold_mismatches,case_mismatches,line_mismatches,shared_prior_verified
0,90kv,1E,window,C90-1E,L90-1E,9022,9022,9022,0,0,0.0,0.0,0,0,0,True
1,90kv,1E,event,C90-1E,L90-1E,9022,9022,9022,0,0,0.0,0.0,0,0,0,True
2,110kv,1E,window,C110-1E,L110-1E,3648,3648,3648,0,0,0.0,0.0,0,0,0,True
3,110kv,1E,event,C110-1E,L110-1E,912,912,912,0,0,0.0,0.0,0,0,0,True
4,90kv,2E,window,C90-2E,L90-2E,9022,9022,9022,0,0,0.0,0.0,0,0,0,True
5,90kv,2E,event,C90-2E,L90-2E,9022,9022,9022,0,0,0.0,0.0,0,0,0,True
6,110kv,2E,window,C110-2E,L110-2E,3648,3648,3648,0,0,0.0,0.0,0,0,0,True
7,110kv,2E,event,C110-2E,L110-2E,912,912,912,0,0,0.0,0.0,0,0,0,True


Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/02_correction_learning_prior_identity_audit.csv


In [11]:
unified_audit = pd.concat(
    [
        window_audit,
        event_audit,
    ],
    ignore_index=True,
    sort=False,
)

unified_audit.to_csv(
    UNIFIED_AUDIT_PATH,
    index=False,
)


shared_prior_summary = (
    prior_identity_audit.loc[
        prior_identity_audit["level"]
        == "event",
        [
            "topology",
            "prior_view",
            "shared_prior_verified",
            "prior_max_difference",
        ],
    ]
    .to_dict(orient="records")
)


mapping_record = {
    "record_version": (
        "chapter4_unified_predictions_v1"
    ),
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "integrity_status": integrity_status,
    "window_table": {
        "path": str(
            UNIFIED_WINDOW_PATH
        ),
        "rows": int(
            len(unified_window)
        ),
        "columns": list(
            unified_window.columns
        ),
        "level_definition": (
            "One row per retained test window."
        ),
    },
    "event_table": {
        "path": str(
            UNIFIED_EVENT_PATH
        ),
        "rows": int(
            len(unified_event)
        ),
        "columns": list(
            unified_event.columns
        ),
        "level_definition": (
            "One row per physical event. "
            "The 110 kV prediction is the arithmetic "
            "mean of four final window predictions; "
            "the 90 kV prediction equals its single "
            "retained prediction."
        ),
    },
    "error_definition": {
        "model_abs_error_pp": (
            "abs(y_pred_pct - y_true_pct)"
        ),
        "prior_abs_error_pp": (
            "abs(y_prior_pct - y_true_pct)"
        ),
        "error_change_pp": (
            "prior_abs_error_pp - "
            "model_abs_error_pp; positive means "
            "the model improved over the prior"
        ),
        "event_averaging_gain_pp": (
            "mean window model absolute error "
            "minus event-mean model absolute error; "
            "positive means averaging helped"
        ),
    },
    "location_bins": (
        "[0,10), [10,20), ..., [90,100], "
        "with 100 assigned to 90-100"
    ),
    "experiments": EXPERIMENTS,
    "correction_learning_shared_prior": (
        shared_prior_summary
    ),
    "optional_columns": {
        "alpha": (
            "Preserved where saved; otherwise NaN."
        ),
        "residual": (
            "Preserved where saved; otherwise NaN."
        ),
        "d_prior_internal": (
            "Preserved for provenance only. "
            "y_prior_pct remains authoritative."
        ),
    },
}


with MAPPING_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mapping_record,
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


print("Saved:", UNIFIED_AUDIT_PATH)
print("Saved:", MAPPING_RECORD_PATH)

display(unified_audit)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/02_unified_prediction_table_audit.csv
Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_prediction_mapping_record.json


,level,experiment_id,rows,expected_rows,events,expected_events,windows_per_event_min,windows_per_event_max,expected_windows_per_event,duplicate_keys,nonfinite_required_rows,mapping_ok,window_rows_min,window_rows_max,events_with_target_spread,events_in_multiple_folds,events_with_case_conflict,events_with_line_conflict,nonfinite_required_events,aggregation_ok
0,window,C90-1E,9022.0,9022.0,9022,9022,1.0,1.0,1,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,window,L90-1E,9022.0,9022.0,9022,9022,1.0,1.0,1,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,window,C110-1E,3648.0,3648.0,912,912,4.0,4.0,4,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,window,L110-1E,3648.0,3648.0,912,912,4.0,4.0,4,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,window,C90-2E,9022.0,9022.0,9022,9022,1.0,1.0,1,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,window,L90-2E,9022.0,9022.0,9022,9022,1.0,1.0,1,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,window,C110-2E,3648.0,3648.0,912,912,4.0,4.0,4,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,window,L110-2E,3648.0,3648.0,912,912,4.0,4.0,4,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,event,C90-1E,NaN,NaN,9022,9022,NaN,NaN,1,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,0.0,0.0,True
9,event,L90-1E,NaN,NaN,9022,9022,NaN,NaN,1,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,0.0,0.0,True


In [12]:
completion_summary = {
    "notebook": (
        "02_posthoc_build_unified_predictions"
    ),
    "status": "PASS",
    "integrity_gate": integrity_status,
    "unified_window_rows": int(
        len(unified_window)
    ),
    "unified_event_rows": int(
        len(unified_event)
    ),
    "experiments": EXPERIMENT_ORDER,
    "window_table": str(
        UNIFIED_WINDOW_PATH
    ),
    "event_table": str(
        UNIFIED_EVENT_PATH
    ),
    "prior_identity_audit": str(
        PRIOR_IDENTITY_AUDIT_PATH
    ),
    "all_window_mappings_passed": bool(
        window_audit["mapping_ok"].all()
    ),
    "all_event_aggregations_passed": bool(
        event_audit[
            "aggregation_ok"
        ].all()
    ),
    "all_aggregation_verifications_passed": bool(
        aggregation_verification[
            "verification_ok"
        ].all()
    ),
    "shared_prior_event_pairs": int(
        prior_identity_audit.loc[
            prior_identity_audit["level"]
            == "event",
            "shared_prior_verified",
        ].sum()
    ),
    "total_prior_event_pairs": 4,
    "next_notebook": (
        "03_posthoc_paired_analysis"
    ),
}


with COMPLETION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        completion_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("=" * 80)
print("NOTEBOOK 02 COMPLETE")
print("=" * 80)

print(
    "Integrity gate:",
    integrity_status,
)

print(
    "Unified window rows:",
    f"{len(unified_window):,}",
)

print(
    "Unified event rows:",
    f"{len(unified_event):,}",
)

print(
    "Window mappings passed:",
    bool(
        window_audit["mapping_ok"].all()
    ),
)

print(
    "Event aggregations passed:",
    bool(
        event_audit[
            "aggregation_ok"
        ].all()
    ),
)

print(
    "Aggregation verification passed:",
    bool(
        aggregation_verification[
            "verification_ok"
        ].all()
    ),
)

print(
    "C/L event-level shared priors verified:",
    completion_summary[
        "shared_prior_event_pairs"
    ],
    "/ 4",
)

print(
    "\nUnified window table:",
    UNIFIED_WINDOW_PATH,
)

print(
    "Unified event table:",
    UNIFIED_EVENT_PATH,
)

print(
    "\nREADY FOR NOTEBOOK 03: "
    "paired prior-versus-model analysis."
)

NOTEBOOK 02 COMPLETE
Integrity gate: PASS
Unified window rows: 50,680
Unified event rows: 39,736
Window mappings passed: True
Event aggregations passed: True
Aggregation verification passed: True
C/L event-level shared priors verified: 4 / 4

Unified window table: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_window_predictions.parquet
Unified event table: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_event_predictions.parquet

READY FOR NOTEBOOK 03: paired prior-versus-model analysis.


In [13]:
from __future__ import annotations

import json
import math

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from IPython.display import display


# =====================================================================
# Project paths
# =====================================================================

THESIS_DIR = Path(
    "/home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS"
)

CHAPTER4_ROOT = THESIS_DIR / "outputs" / "chapter4"

FINAL_DIR = CHAPTER4_ROOT / "posthoc_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# =====================================================================
# Required inputs
# =====================================================================

INTEGRITY_JSON_PATH = (
    FINAL_DIR
    / "posthoc_integrity_audit.json"
)

UNIFIED_WINDOW_PATH = (
    FINAL_DIR
    / "posthoc_unified_window_predictions.parquet"
)

UNIFIED_EVENT_PATH = (
    FINAL_DIR
    / "posthoc_unified_event_predictions.parquet"
)


# =====================================================================
# Outputs
# =====================================================================

PAIRED_OUTCOME_EVENT_PATH = (
    FINAL_DIR
    / "posthoc_event_paired_outcomes.parquet"
)

PAIRED_OUTCOME_WINDOW_PATH = (
    FINAL_DIR
    / "posthoc_window_paired_outcomes.parquet"
)

PAIRED_METRICS_PATH = (
    FINAL_DIR
    / "posthoc_paired_prior_comparison.csv"
)

EVENT_HEADLINE_PATH = (
    FINAL_DIR
    / "posthoc_event_headline_metrics.csv"
)

EPS_CATEGORY_PATH = (
    FINAL_DIR
    / "posthoc_eps_category_counts.csv"
)

BOOTSTRAP_PATH = (
    FINAL_DIR
    / "posthoc_event_bootstrap_intervals.csv"
)

CORRECTION_LEARNING_PATH = (
    FINAL_DIR
    / "posthoc_correction_vs_learning.csv"
)

COMPLETION_PATH = (
    FINAL_DIR
    / "03_paired_analysis_completion_summary.json"
)


# =====================================================================
# Analysis settings
# =====================================================================

EPSILON_PP = 1e-8

BOOTSTRAP_REPLICATES = 10_000
BOOTSTRAP_SEED = 42
BOOTSTRAP_CHUNK_SIZE = 250

CI_LOWER_QUANTILE = 0.025
CI_UPPER_QUANTILE = 0.975


EXPERIMENT_ORDER = [
    "C90-1E",
    "L90-1E",
    "C110-1E",
    "L110-1E",
    "C90-2E",
    "L90-2E",
    "C110-2E",
    "L110-2E",
]


EXPECTED_EVENT_ROWS = {
    "C90-1E": 9022,
    "L90-1E": 9022,
    "C110-1E": 912,
    "L110-1E": 912,
    "C90-2E": 9022,
    "L90-2E": 9022,
    "C110-2E": 912,
    "L110-2E": 912,
}


EXPECTED_WINDOW_ROWS = {
    "C90-1E": 9022,
    "L90-1E": 9022,
    "C110-1E": 3648,
    "L110-1E": 3648,
    "C90-2E": 9022,
    "L90-2E": 9022,
    "C110-2E": 3648,
    "L110-2E": 3648,
}


for required_path in [
    INTEGRITY_JSON_PATH,
    UNIFIED_WINDOW_PATH,
    UNIFIED_EVENT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


print("Unified event input:", UNIFIED_EVENT_PATH)
print("Unified window input:", UNIFIED_WINDOW_PATH)
print("Bootstrap replicates:", BOOTSTRAP_REPLICATES)
print("Bootstrap seed:", BOOTSTRAP_SEED)

Unified event input: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_event_predictions.parquet
Unified window input: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_window_predictions.parquet
Bootstrap replicates: 10000
Bootstrap seed: 42


In [14]:
with INTEGRITY_JSON_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    integrity_payload = json.load(file)


if integrity_payload.get("overall_status") != "PASS":
    raise RuntimeError(
        "The mandatory integrity audit did not pass."
    )


unified_event = pd.read_parquet(
    UNIFIED_EVENT_PATH
)

unified_window = pd.read_parquet(
    UNIFIED_WINDOW_PATH
)


print("Unified event shape:", unified_event.shape)
print("Unified window shape:", unified_window.shape)


assert len(unified_event) == 39_736
assert len(unified_window) == 50_680


required_columns = {
    "experiment_id",
    "topology",
    "prior_view",
    "model_family",
    "sample_id",
    "fold",
    "y_true_pct",
    "y_pred_pct",
    "y_prior_pct",
    "model_abs_error_pp",
    "prior_abs_error_pp",
    "error_change_pp",
    "case",
    "broad_fault_family",
    "y_fault_line",
    "location_bin_10pp",
}


for label, frame in {
    "event": unified_event,
    "window": unified_window,
}.items():
    missing = sorted(
        required_columns
        - set(frame.columns)
    )

    if missing:
        raise KeyError(
            f"{label} table is missing: {missing}"
        )


event_counts = (
    unified_event["experiment_id"]
    .value_counts()
    .reindex(EXPERIMENT_ORDER)
)

window_counts = (
    unified_window["experiment_id"]
    .value_counts()
    .reindex(EXPERIMENT_ORDER)
)


for experiment_id in EXPERIMENT_ORDER:
    assert (
        int(event_counts[experiment_id])
        == EXPECTED_EVENT_ROWS[experiment_id]
    )

    assert (
        int(window_counts[experiment_id])
        == EXPECTED_WINDOW_ROWS[experiment_id]
    )


print("\nIntegrity and cohort gates passed.")

display(
    pd.DataFrame(
        {
            "event_rows": event_counts,
            "window_rows": window_counts,
        }
    )
)

Unified event shape: (39736, 54)
Unified window shape: (50680, 30)

Integrity and cohort gates passed.


,event_rows,window_rows
experiment_id,,
C90-1E,9022,9022
L90-1E,9022,9022
C110-1E,912,3648
L110-1E,912,3648
C90-2E,9022,9022
L90-2E,9022,9022
C110-2E,912,3648
L110-2E,912,3648


In [15]:
def finite_numeric(
    values: pd.Series,
) -> np.ndarray:
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    return numeric[
        np.isfinite(numeric)
    ]


def safe_percentage(
    numerator: float,
    denominator: float,
) -> float:
    if not np.isfinite(denominator):
        return np.nan

    if abs(denominator) <= EPSILON_PP:
        return np.nan

    return float(
        100.0 * numerator / denominator
    )


def percentile(
    values: np.ndarray,
    quantile: float,
) -> float:
    if len(values) == 0:
        return np.nan

    return float(
        np.quantile(
            values,
            quantile,
        )
    )


def wilson_interval(
    success_count: int,
    sample_count: int,
    z_value: float = 1.959963984540054,
) -> tuple[float, float]:
    if sample_count <= 0:
        return np.nan, np.nan

    proportion = success_count / sample_count

    denominator = (
        1.0
        + z_value**2 / sample_count
    )

    centre = (
        proportion
        + z_value**2
        / (2.0 * sample_count)
    ) / denominator

    radius = (
        z_value
        * math.sqrt(
            (
                proportion
                * (1.0 - proportion)
                / sample_count
            )
            + (
                z_value**2
                / (4.0 * sample_count**2)
            )
        )
        / denominator
    )

    return (
        100.0 * (centre - radius),
        100.0 * (centre + radius),
    )


def add_paired_outcomes(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()

    output["paired_gain_pp"] = (
        output["prior_abs_error_pp"]
        - output["model_abs_error_pp"]
    )

    output["eps_outcome"] = np.select(
        [
            output["paired_gain_pp"]
            > EPSILON_PP,

            output["paired_gain_pp"]
            < -EPSILON_PP,
        ],
        [
            "improved",
            "worsened",
        ],
        default="tied_within_epsilon",
    )

    output["material_1pp_outcome"] = np.select(
        [
            output["paired_gain_pp"] >= 1.0,
            output["paired_gain_pp"] <= -1.0,
        ],
        [
            "improved_by_at_least_1pp",
            "worsened_by_at_least_1pp",
        ],
        default="change_within_1pp",
    )

    output["material_5pp_outcome"] = np.select(
        [
            output["paired_gain_pp"] >= 5.0,
            output["paired_gain_pp"] <= -5.0,
        ],
        [
            "improved_by_at_least_5pp",
            "worsened_by_at_least_5pp",
        ],
        default="change_within_5pp",
    )

    return output


def summarize_paired_group(
    frame: pd.DataFrame,
    level: str,
) -> dict[str, Any]:
    experiment_ids = (
        frame["experiment_id"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if len(experiment_ids) != 1:
        raise RuntimeError(
            "Expected one experiment per group."
        )

    experiment_id = experiment_ids[0]

    target = finite_numeric(
        frame["y_true_pct"]
    )

    prediction = finite_numeric(
        frame["y_pred_pct"]
    )

    prior = finite_numeric(
        frame["y_prior_pct"]
    )

    model_error = finite_numeric(
        frame["model_abs_error_pp"]
    )

    prior_error = finite_numeric(
        frame["prior_abs_error_pp"]
    )

    gain = finite_numeric(
        frame["paired_gain_pp"]
    )

    n_rows = len(frame)

    if not (
        len(target)
        == len(prediction)
        == len(prior)
        == len(model_error)
        == len(prior_error)
        == len(gain)
        == n_rows
    ):
        raise RuntimeError(
            f"Non-finite paired data in {experiment_id}."
        )

    improved_count = int(
        np.sum(gain > EPSILON_PP)
    )

    worsened_count = int(
        np.sum(gain < -EPSILON_PP)
    )

    tied_count = int(
        n_rows
        - improved_count
        - worsened_count
    )

    improved_ci_lower, improved_ci_upper = (
        wilson_interval(
            improved_count,
            n_rows,
        )
    )

    prior_mae = float(
        np.mean(prior_error)
    )

    model_mae = float(
        np.mean(model_error)
    )

    mae_reduction = (
        prior_mae - model_mae
    )

    return {
        "level": level,
        "experiment_id": experiment_id,
        "topology": str(
            frame["topology"].iloc[0]
        ),
        "prior_view": str(
            frame["prior_view"].iloc[0]
        ),
        "model_family": str(
            frame["model_family"].iloc[0]
        ),
        "n": n_rows,

        "prior_mae_pp": prior_mae,
        "model_mae_pp": model_mae,
        "mae_reduction_pp": mae_reduction,
        "relative_mae_reduction_pct": (
            safe_percentage(
                mae_reduction,
                prior_mae,
            )
        ),

        "prior_rmse_pp": float(
            np.sqrt(
                np.mean(
                    (prior - target) ** 2
                )
            )
        ),

        "model_rmse_pp": float(
            np.sqrt(
                np.mean(
                    (prediction - target) ** 2
                )
            )
        ),

        "prior_median_ae_pp": float(
            np.median(prior_error)
        ),

        "model_median_ae_pp": float(
            np.median(model_error)
        ),

        "mean_paired_gain_pp": float(
            np.mean(gain)
        ),

        "median_paired_gain_pp": float(
            np.median(gain)
        ),

        "paired_gain_p10_pp": percentile(
            gain,
            0.10,
        ),

        "paired_gain_p90_pp": percentile(
            gain,
            0.90,
        ),

        "model_bias_pp": float(
            np.mean(
                prediction - target
            )
        ),

        "prior_bias_pp": float(
            np.mean(
                prior - target
            )
        ),

        "improved_count": improved_count,
        "worsened_count": worsened_count,
        "tied_count": tied_count,

        "improved_rate_pct": (
            100.0
            * improved_count
            / n_rows
        ),

        "improved_rate_ci95_lower_pct": (
            improved_ci_lower
        ),

        "improved_rate_ci95_upper_pct": (
            improved_ci_upper
        ),

        "worsened_rate_pct": (
            100.0
            * worsened_count
            / n_rows
        ),

        "tied_rate_pct": (
            100.0
            * tied_count
            / n_rows
        ),

        "net_improvement_rate_pp": (
            100.0
            * (
                improved_count
                - worsened_count
            )
            / n_rows
        ),

        "improved_ge_1pp_rate_pct": (
            100.0
            * np.mean(gain >= 1.0)
        ),

        "worsened_ge_1pp_rate_pct": (
            100.0
            * np.mean(gain <= -1.0)
        ),

        "improved_ge_5pp_rate_pct": (
            100.0
            * np.mean(gain >= 5.0)
        ),

        "worsened_ge_5pp_rate_pct": (
            100.0
            * np.mean(gain <= -5.0)
        ),
    }


def paired_bootstrap_mean_gain(
    gain_values: np.ndarray,
    rng: np.random.Generator,
    replicates: int = BOOTSTRAP_REPLICATES,
    chunk_size: int = BOOTSTRAP_CHUNK_SIZE,
) -> np.ndarray:
    values = np.asarray(
        gain_values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    sample_count = len(values)

    if sample_count == 0:
        raise ValueError(
            "No finite paired gains."
        )

    estimates = np.empty(
        replicates,
        dtype=float,
    )

    completed = 0

    while completed < replicates:
        current_chunk = min(
            chunk_size,
            replicates - completed,
        )

        indices = rng.integers(
            low=0,
            high=sample_count,
            size=(
                current_chunk,
                sample_count,
            ),
            endpoint=False,
        )

        sampled_values = values[indices]

        estimates[
            completed:
            completed + current_chunk
        ] = sampled_values.mean(axis=1)

        completed += current_chunk

    return estimates

In [16]:
event_outcomes = add_paired_outcomes(
    unified_event
)

window_outcomes = add_paired_outcomes(
    unified_window
)


event_identity_difference = (
    event_outcomes["paired_gain_pp"]
    - event_outcomes["error_change_pp"]
).abs()

window_identity_difference = (
    window_outcomes["paired_gain_pp"]
    - window_outcomes["error_change_pp"]
).abs()


assert (
    event_identity_difference
    <= EPSILON_PP
).all()

assert (
    window_identity_difference
    <= EPSILON_PP
).all()


event_outcomes.to_parquet(
    PAIRED_OUTCOME_EVENT_PATH,
    index=False,
)

window_outcomes.to_parquet(
    PAIRED_OUTCOME_WINDOW_PATH,
    index=False,
)


print(
    "Saved event paired outcomes:",
    PAIRED_OUTCOME_EVENT_PATH,
)

print(
    "Saved window paired outcomes:",
    PAIRED_OUTCOME_WINDOW_PATH,
)


display(
    event_outcomes[
        [
            "experiment_id",
            "sample_id",
            "prior_abs_error_pp",
            "model_abs_error_pp",
            "paired_gain_pp",
            "eps_outcome",
        ]
    ].head()
)

Saved event paired outcomes: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_event_paired_outcomes.parquet
Saved window paired outcomes: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_window_paired_outcomes.parquet


,experiment_id,sample_id,prior_abs_error_pp,model_abs_error_pp,paired_gain_pp,eps_outcome
0,C90-1E,0,3.678651,0.990178,2.688473,improved
1,C90-1E,1,12.435049,25.316566,-12.881517,worsened
2,C90-1E,10,7.337058,3.785181,3.551877,improved
3,C90-1E,100,1.355839,1.124918,0.230920,improved
4,C90-1E,1000,12.662756,6.741834,5.920923,improved


In [17]:
paired_summary_rows = []


for level, source in [
    ("event", event_outcomes),
    ("window", window_outcomes),
]:
    for experiment_id in EXPERIMENT_ORDER:
        experiment_frame = source.loc[
            source["experiment_id"]
            == experiment_id
        ].copy()

        paired_summary_rows.append(
            summarize_paired_group(
                experiment_frame,
                level,
            )
        )


paired_metrics = pd.DataFrame(
    paired_summary_rows
)


experiment_order_map = {
    experiment_id: index
    for index, experiment_id
    in enumerate(EXPERIMENT_ORDER)
}


paired_metrics["_experiment_order"] = (
    paired_metrics["experiment_id"].map(
        experiment_order_map
    )
)

paired_metrics["_level_order"] = (
    paired_metrics["level"].map(
        {
            "event": 0,
            "window": 1,
        }
    )
)

paired_metrics = (
    paired_metrics
    .sort_values(
        [
            "_level_order",
            "_experiment_order",
        ],
        kind="stable",
    )
    .drop(
        columns=[
            "_level_order",
            "_experiment_order",
        ]
    )
    .reset_index(drop=True)
)


paired_metrics.to_csv(
    PAIRED_METRICS_PATH,
    index=False,
)


eps_category_counts = (
    pd.concat(
        [
            event_outcomes.assign(
                level="event"
            ),
            window_outcomes.assign(
                level="window"
            ),
        ],
        ignore_index=True,
    )
    .groupby(
        [
            "level",
            "experiment_id",
            "eps_outcome",
        ],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)


eps_totals = (
    eps_category_counts.groupby(
        [
            "level",
            "experiment_id",
        ]
    )["count"]
    .transform("sum")
)


eps_category_counts[
    "rate_pct"
] = (
    100.0
    * eps_category_counts["count"]
    / eps_totals
)


eps_category_counts.to_csv(
    EPS_CATEGORY_PATH,
    index=False,
)


print("Saved:", PAIRED_METRICS_PATH)
print("Saved:", EPS_CATEGORY_PATH)

display(
    paired_metrics.loc[
        paired_metrics["level"]
        == "event"
    ]
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_paired_prior_comparison.csv
Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_eps_category_counts.csv


,level,experiment_id,topology,prior_view,model_family,n,prior_mae_pp,model_mae_pp,mae_reduction_pp,relative_mae_reduction_pct,...,improved_rate_pct,improved_rate_ci95_lower_pct,improved_rate_ci95_upper_pct,worsened_rate_pct,tied_rate_pct,net_improvement_rate_pp,improved_ge_1pp_rate_pct,worsened_ge_1pp_rate_pct,improved_ge_5pp_rate_pct,worsened_ge_5pp_rate_pct
0,event,C90-1E,90kv,1E,correction,9022,7.322130,6.265098,1.057032,14.436127,...,43.526934,42.506854,44.552524,30.203946,26.269120,13.322988,33.750831,21.081800,13.666593,5.796941
1,event,L90-1E,90kv,1E,combination,9022,7.322130,6.288006,1.034124,14.123271,...,45.034360,44.010059,46.062888,28.962536,26.003104,16.071824,34.305032,18.199956,11.893150,3.624474
2,event,C110-1E,110kv,1E,correction,912,11.692447,6.636678,5.055769,43.239611,...,43.750000,40.563275,46.989156,22.697368,33.552632,21.052632,36.403509,14.144737,25.438596,8.004386
3,event,L110-1E,110kv,1E,combination,912,11.692447,7.825728,3.866719,33.070232,...,48.355263,45.125675,51.598649,23.026316,28.618421,25.328947,33.881579,11.842105,20.614035,4.934211
4,event,C90-2E,90kv,2E,correction,9022,1.148614,1.347025,-0.198410,-17.273892,...,32.775438,31.814368,33.751169,65.850144,1.374418,-33.074706,1.695855,7.459543,0.110840,0.088672
5,event,L90-2E,90kv,2E,combination,9022,1.148614,1.153115,-0.004500,-0.391818,...,46.696963,45.669109,47.727628,51.928619,1.374418,-5.231656,0.121924,0.011084,0.000000,0.000000
6,event,C110-2E,110kv,2E,correction,912,0.249126,0.468317,-0.219191,-87.983652,...,30.592105,27.688069,33.658952,67.763158,1.644737,-37.171053,0.219298,7.236842,0.000000,0.000000
7,event,L110-2E,110kv,2E,combination,912,0.249126,0.238907,0.010220,4.102232,...,45.942982,42.732400,49.187599,52.412281,1.644737,-6.469298,0.000000,0.000000,0.000000,0.000000


In [18]:
bootstrap_rows = []

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


for experiment_id in EXPERIMENT_ORDER:
    experiment_frame = event_outcomes.loc[
        event_outcomes["experiment_id"]
        == experiment_id
    ].copy()

    gain = finite_numeric(
        experiment_frame["paired_gain_pp"]
    )

    bootstrap_estimates = (
        paired_bootstrap_mean_gain(
            gain_values=gain,
            rng=rng,
            replicates=BOOTSTRAP_REPLICATES,
            chunk_size=BOOTSTRAP_CHUNK_SIZE,
        )
    )

    observed_mean_gain = float(
        np.mean(gain)
    )

    ci_lower = float(
        np.quantile(
            bootstrap_estimates,
            CI_LOWER_QUANTILE,
        )
    )

    ci_upper = float(
        np.quantile(
            bootstrap_estimates,
            CI_UPPER_QUANTILE,
        )
    )

    bootstrap_standard_error = float(
        np.std(
            bootstrap_estimates,
            ddof=1,
        )
    )

    bootstrap_probability_positive = float(
        np.mean(
            bootstrap_estimates > 0.0
        )
    )

    bootstrap_probability_negative = float(
        np.mean(
            bootstrap_estimates < 0.0
        )
    )

    if ci_lower > 0.0:
        conclusion = (
            "bootstrap_interval_above_zero"
        )

    elif ci_upper < 0.0:
        conclusion = (
            "bootstrap_interval_below_zero"
        )

    else:
        conclusion = (
            "bootstrap_interval_crosses_zero"
        )

    bootstrap_rows.append(
        {
            "experiment_id": experiment_id,
            "n_events": len(gain),
            "bootstrap_replicates": (
                BOOTSTRAP_REPLICATES
            ),
            "bootstrap_seed": (
                BOOTSTRAP_SEED
            ),
            "observed_mean_gain_pp": (
                observed_mean_gain
            ),
            "bootstrap_mean_gain_pp": float(
                np.mean(
                    bootstrap_estimates
                )
            ),
            "bootstrap_standard_error_pp": (
                bootstrap_standard_error
            ),
            "ci95_lower_pp": ci_lower,
            "ci95_upper_pp": ci_upper,
            "bootstrap_probability_gain_positive": (
                bootstrap_probability_positive
            ),
            "bootstrap_probability_gain_negative": (
                bootstrap_probability_negative
            ),
            "bootstrap_conclusion": conclusion,
        }
    )

    print(
        f"Completed bootstrap: {experiment_id}"
    )


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


bootstrap_results.to_csv(
    BOOTSTRAP_PATH,
    index=False,
)


print("\nSaved:", BOOTSTRAP_PATH)

display(bootstrap_results)

Completed bootstrap: C90-1E
Completed bootstrap: L90-1E
Completed bootstrap: C110-1E
Completed bootstrap: L110-1E
Completed bootstrap: C90-2E
Completed bootstrap: L90-2E
Completed bootstrap: C110-2E
Completed bootstrap: L110-2E

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_event_bootstrap_intervals.csv


,experiment_id,n_events,bootstrap_replicates,bootstrap_seed,observed_mean_gain_pp,bootstrap_mean_gain_pp,bootstrap_standard_error_pp,ci95_lower_pp,ci95_upper_pp,bootstrap_probability_gain_positive,bootstrap_probability_gain_negative,bootstrap_conclusion
0,C90-1E,9022,10000,42,1.057032,1.057369,0.060255,0.938860,1.172843,1.0000,0.0000,bootstrap_interval_above_zero
1,L90-1E,9022,10000,42,1.034124,1.034430,0.044106,0.948886,1.121217,1.0000,0.0000,bootstrap_interval_above_zero
2,C110-1E,912,10000,42,5.055769,5.051154,0.434336,4.208828,5.905435,1.0000,0.0000,bootstrap_interval_above_zero
3,L110-1E,912,10000,42,3.866719,3.863209,0.365112,3.162477,4.597567,1.0000,0.0000,bootstrap_interval_above_zero
4,C90-2E,9022,10000,42,-0.198410,-0.198420,0.008791,-0.215734,-0.180964,0.0000,1.0000,bootstrap_interval_below_zero
5,L90-2E,9022,10000,42,-0.004500,-0.004501,0.001269,-0.007002,-0.002001,0.0003,0.9997,bootstrap_interval_below_zero
6,C110-2E,912,10000,42,-0.219191,-0.219215,0.015108,-0.249709,-0.190191,0.0000,1.0000,bootstrap_interval_below_zero
7,L110-2E,912,10000,42,0.010220,0.010312,0.004216,0.002226,0.018720,0.9944,0.0056,bootstrap_interval_above_zero


In [19]:
event_metrics = paired_metrics.loc[
    paired_metrics["level"] == "event"
].copy()


event_headline = event_metrics.merge(
    bootstrap_results,
    on="experiment_id",
    how="left",
    validate="one_to_one",
)


event_headline[
    "mae_reduction_identity_difference"
] = (
    event_headline["mae_reduction_pp"]
    - event_headline[
        "observed_mean_gain_pp"
    ]
).abs()


if not (
    event_headline[
        "mae_reduction_identity_difference"
    ]
    <= EPSILON_PP
).all():
    raise RuntimeError(
        "MAE reduction and mean paired gain disagree."
    )


event_headline[
    "result_direction"
] = np.select(
    [
        event_headline[
            "ci95_lower_pp"
        ] > 0.0,

        event_headline[
            "ci95_upper_pp"
        ] < 0.0,
    ],
    [
        "improved_over_prior",
        "worsened_relative_to_prior",
    ],
    default="uncertain_average_change",
)


event_headline["_experiment_order"] = (
    event_headline["experiment_id"].map(
        experiment_order_map
    )
)

event_headline = (
    event_headline
    .sort_values(
        "_experiment_order",
        kind="stable",
    )
    .drop(columns="_experiment_order")
    .reset_index(drop=True)
)


event_headline.to_csv(
    EVENT_HEADLINE_PATH,
    index=False,
)


print("Saved:", EVENT_HEADLINE_PATH)


headline_display_columns = [
    "experiment_id",
    "n",
    "prior_mae_pp",
    "model_mae_pp",
    "mae_reduction_pp",
    "relative_mae_reduction_pct",
    "improved_rate_pct",
    "worsened_rate_pct",
    "improved_ge_5pp_rate_pct",
    "worsened_ge_5pp_rate_pct",
    "ci95_lower_pp",
    "ci95_upper_pp",
    "result_direction",
]


display(
    event_headline[
        headline_display_columns
    ]
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_event_headline_metrics.csv


,experiment_id,n,prior_mae_pp,model_mae_pp,mae_reduction_pp,relative_mae_reduction_pct,improved_rate_pct,worsened_rate_pct,improved_ge_5pp_rate_pct,worsened_ge_5pp_rate_pct,ci95_lower_pp,ci95_upper_pp,result_direction
0,C90-1E,9022,7.322130,6.265098,1.057032,14.436127,43.526934,30.203946,13.666593,5.796941,0.938860,1.172843,improved_over_prior
1,L90-1E,9022,7.322130,6.288006,1.034124,14.123271,45.034360,28.962536,11.893150,3.624474,0.948886,1.121217,improved_over_prior
2,C110-1E,912,11.692447,6.636678,5.055769,43.239611,43.750000,22.697368,25.438596,8.004386,4.208828,5.905435,improved_over_prior
3,L110-1E,912,11.692447,7.825728,3.866719,33.070232,48.355263,23.026316,20.614035,4.934211,3.162477,4.597567,improved_over_prior
4,C90-2E,9022,1.148614,1.347025,-0.198410,-17.273892,32.775438,65.850144,0.110840,0.088672,-0.215734,-0.180964,worsened_relative_to_prior
5,L90-2E,9022,1.148614,1.153115,-0.004500,-0.391818,46.696963,51.928619,0.000000,0.000000,-0.007002,-0.002001,worsened_relative_to_prior
6,C110-2E,912,0.249126,0.468317,-0.219191,-87.983652,30.592105,67.763158,0.000000,0.000000,-0.249709,-0.190191,worsened_relative_to_prior
7,L110-2E,912,0.249126,0.238907,0.010220,4.102232,45.942982,52.412281,0.000000,0.000000,0.002226,0.018720,improved_over_prior


In [20]:
PAIR_DEFINITIONS = [
    {
        "topology": "90kv",
        "prior_view": "1E",
        "correction": "C90-1E",
        "combination": "L90-1E",
        "expected_events": 9022,
    },
    {
        "topology": "110kv",
        "prior_view": "1E",
        "correction": "C110-1E",
        "combination": "L110-1E",
        "expected_events": 912,
    },
    {
        "topology": "90kv",
        "prior_view": "2E",
        "correction": "C90-2E",
        "combination": "L90-2E",
        "expected_events": 9022,
    },
    {
        "topology": "110kv",
        "prior_view": "2E",
        "correction": "C110-2E",
        "combination": "L110-2E",
        "expected_events": 912,
    },
]


correction_learning_rows = []


for pair in PAIR_DEFINITIONS:
    correction = event_outcomes.loc[
        event_outcomes["experiment_id"]
        == pair["correction"],
        [
            "sample_id",
            "fold",
            "y_true_pct",
            "y_prior_pct",
            "model_abs_error_pp",
        ],
    ].copy()

    combination = event_outcomes.loc[
        event_outcomes["experiment_id"]
        == pair["combination"],
        [
            "sample_id",
            "fold",
            "y_true_pct",
            "y_prior_pct",
            "model_abs_error_pp",
        ],
    ].copy()

    merged = correction.merge(
        combination,
        on="sample_id",
        how="outer",
        suffixes=(
            "_correction",
            "_combination",
        ),
        indicator=True,
        validate="one_to_one",
    )

    unmatched_count = int(
        (merged["_merge"] != "both").sum()
    )

    matched = merged.loc[
        merged["_merge"] == "both"
    ].copy()

    target_difference = (
        matched[
            "y_true_pct_correction"
        ]
        - matched[
            "y_true_pct_combination"
        ]
    ).abs()

    prior_difference = (
        matched[
            "y_prior_pct_correction"
        ]
        - matched[
            "y_prior_pct_combination"
        ]
    ).abs()

    fold_equal_mask = (
        matched["fold_correction"]
        .astype("Int64")
        .eq(
            matched["fold_combination"]
            .astype("Int64")
        )
    )

    fold_mismatch_count = int(
        (~fold_equal_mask).sum()
    )

    pair_ok = bool(
        unmatched_count == 0
        and len(matched)
        == pair["expected_events"]
        and (
            target_difference
            <= EPSILON_PP
        ).all()
        and (
            prior_difference
            <= EPSILON_PP
        ).all()
        and fold_mismatch_count == 0
    )

    if not pair_ok:
        raise RuntimeError(
            "Correction/combination pairing failed for "
            f"{pair['topology']} {pair['prior_view']}."
        )

    correction_error = pd.to_numeric(
        matched[
            "model_abs_error_pp_correction"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    combination_error = pd.to_numeric(
        matched[
            "model_abs_error_pp_combination"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    # Positive means combination learning is better.
    learning_gain = (
        correction_error
        - combination_error
    )

    learning_better_count = int(
        np.sum(
            learning_gain > EPSILON_PP
        )
    )

    correction_better_count = int(
        np.sum(
            learning_gain < -EPSILON_PP
        )
    )

    tied_count = int(
        len(learning_gain)
        - learning_better_count
        - correction_better_count
    )

    correction_learning_rows.append(
        {
            "topology": pair["topology"],
            "prior_view": pair["prior_view"],
            "correction_experiment": (
                pair["correction"]
            ),
            "combination_experiment": (
                pair["combination"]
            ),
            "n_events": len(
                learning_gain
            ),
            "correction_mae_pp": float(
                np.mean(
                    correction_error
                )
            ),
            "combination_mae_pp": float(
                np.mean(
                    combination_error
                )
            ),
            "combination_mae_gain_pp": float(
                np.mean(
                    learning_gain
                )
            ),
            "combination_relative_mae_gain_pct": (
                safe_percentage(
                    float(
                        np.mean(
                            learning_gain
                        )
                    ),
                    float(
                        np.mean(
                            correction_error
                        )
                    ),
                )
            ),
            "combination_better_count": (
                learning_better_count
            ),
            "correction_better_count": (
                correction_better_count
            ),
            "tied_count": tied_count,
            "combination_better_rate_pct": (
                100.0
                * learning_better_count
                / len(learning_gain)
            ),
            "correction_better_rate_pct": (
                100.0
                * correction_better_count
                / len(learning_gain)
            ),
            "tied_rate_pct": (
                100.0
                * tied_count
                / len(learning_gain)
            ),
            "combination_better_ge_1pp_rate_pct": (
                100.0
                * np.mean(
                    learning_gain >= 1.0
                )
            ),
            "correction_better_ge_1pp_rate_pct": (
                100.0
                * np.mean(
                    learning_gain <= -1.0
                )
            ),
            "median_combination_gain_pp": float(
                np.median(
                    learning_gain
                )
            ),
            "combination_gain_p10_pp": float(
                np.quantile(
                    learning_gain,
                    0.10,
                )
            ),
            "combination_gain_p90_pp": float(
                np.quantile(
                    learning_gain,
                    0.90,
                )
            ),
        }
    )


correction_learning = pd.DataFrame(
    correction_learning_rows
)

correction_learning.to_csv(
    CORRECTION_LEARNING_PATH,
    index=False,
)


print("Saved:", CORRECTION_LEARNING_PATH)

display(correction_learning)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_correction_vs_learning.csv


,topology,prior_view,correction_experiment,combination_experiment,n_events,correction_mae_pp,combination_mae_pp,combination_mae_gain_pp,combination_relative_mae_gain_pct,combination_better_count,correction_better_count,tied_count,combination_better_rate_pct,correction_better_rate_pct,tied_rate_pct,combination_better_ge_1pp_rate_pct,correction_better_ge_1pp_rate_pct,median_combination_gain_pp,combination_gain_p10_pp,combination_gain_p90_pp
0,90kv,1E,C90-1E,L90-1E,9022,6.265098,6.288006,-0.022908,-0.365640,3369,3317,2336,37.342053,36.765684,25.892263,25.338063,24.695190,0.000000,-3.591998,3.623280
1,110kv,1E,C110-1E,L110-1E,912,6.636678,7.825728,-1.189049,-17.916331,262,375,275,28.728070,41.118421,30.153509,17.982456,31.688596,0.000000,-9.483538,4.733918
2,90kv,2E,C90-2E,L90-2E,9022,1.347025,1.153115,0.193910,14.395425,4842,1884,2296,53.668810,20.882288,25.448903,7.193527,1.407670,0.025005,-0.140713,0.798773
3,110kv,2E,C110-2E,L110-2E,912,0.468317,0.238907,0.229410,48.986113,558,174,180,61.184211,19.078947,19.736842,6.798246,0.219298,0.035773,-0.068691,0.824673


In [21]:
print("=" * 100)
print("EVENT-LEVEL MODEL VERSUS MATCHED PRIOR")
print("=" * 100)

print(
    event_headline[
        [
            "experiment_id",
            "n",
            "prior_mae_pp",
            "model_mae_pp",
            "mae_reduction_pp",
            "relative_mae_reduction_pct",
            "improved_rate_pct",
            "worsened_rate_pct",
            "ci95_lower_pp",
            "ci95_upper_pp",
            "result_direction",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


print("\n")
print("=" * 100)
print("EVENT-LEVEL CORRECTION VERSUS COMBINATION LEARNING")
print("=" * 100)

print(
    correction_learning[
        [
            "topology",
            "prior_view",
            "correction_experiment",
            "combination_experiment",
            "correction_mae_pp",
            "combination_mae_pp",
            "combination_mae_gain_pp",
            "combination_better_rate_pct",
            "correction_better_rate_pct",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

EVENT-LEVEL MODEL VERSUS MATCHED PRIOR
experiment_id    n  prior_mae_pp  model_mae_pp  mae_reduction_pp  relative_mae_reduction_pct  improved_rate_pct  worsened_rate_pct  ci95_lower_pp  ci95_upper_pp           result_direction
       C90-1E 9022      7.322130      6.265098          1.057032                   14.436127          43.526934          30.203946       0.938860       1.172843        improved_over_prior
       L90-1E 9022      7.322130      6.288006          1.034124                   14.123271          45.034360          28.962536       0.948886       1.121217        improved_over_prior
      C110-1E  912     11.692447      6.636678          5.055769                   43.239611          43.750000          22.697368       4.208828       5.905435        improved_over_prior
      L110-1E  912     11.692447      7.825728          3.866719                   33.070232          48.355263          23.026316       3.162477       4.597567        improved_over_prior
       C90-2E 9022   

In [22]:
bootstrap_passed = bool(
    len(bootstrap_results)
    == len(EXPERIMENT_ORDER)
    and (
        bootstrap_results[
            "bootstrap_replicates"
        ]
        == BOOTSTRAP_REPLICATES
    ).all()
)


completion_summary = {
    "notebook": (
        "03_posthoc_paired_analysis"
    ),
    "status": "PASS",
    "analysis_unit_primary": (
        "physical_event"
    ),
    "analysis_unit_secondary": (
        "retained_window"
    ),
    "experiments": EXPERIMENT_ORDER,
    "event_rows": int(
        len(event_outcomes)
    ),
    "window_rows": int(
        len(window_outcomes)
    ),
    "bootstrap_replicates_per_experiment": (
        BOOTSTRAP_REPLICATES
    ),
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_completed": (
        bootstrap_passed
    ),
    "event_headline_metrics": str(
        EVENT_HEADLINE_PATH
    ),
    "paired_metrics": str(
        PAIRED_METRICS_PATH
    ),
    "bootstrap_intervals": str(
        BOOTSTRAP_PATH
    ),
    "eps_category_counts": str(
        EPS_CATEGORY_PATH
    ),
    "correction_vs_learning": str(
        CORRECTION_LEARNING_PATH
    ),
    "next_notebook": (
        "04_posthoc_subgroups_and_tails"
    ),
}


with COMPLETION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        completion_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("\n")
print("=" * 80)
print("NOTEBOOK 03 COMPLETE")
print("=" * 80)

print(
    "Status:",
    completion_summary["status"],
)

print(
    "Experiments analyzed:",
    len(EXPERIMENT_ORDER),
)

print(
    "Event-level paired rows:",
    f"{len(event_outcomes):,}",
)

print(
    "Window-level paired rows:",
    f"{len(window_outcomes):,}",
)

print(
    "Bootstrap replicates:",
    BOOTSTRAP_REPLICATES,
    "per experiment",
)

print(
    "Bootstrap seed:",
    BOOTSTRAP_SEED,
)

print(
    "Bootstrap completed:",
    bootstrap_passed,
)

print(
    "\nEvent headline metrics:",
    EVENT_HEADLINE_PATH,
)

print(
    "Correction-versus-learning results:",
    CORRECTION_LEARNING_PATH,
)

print(
    "\nThe main post-hoc narrative can now be drafted."
)

print(
    "READY FOR NOTEBOOK 04: "
    "subgroup and tail analysis."
)



NOTEBOOK 03 COMPLETE
Status: PASS
Experiments analyzed: 8
Event-level paired rows: 39,736
Window-level paired rows: 50,680
Bootstrap replicates: 10000 per experiment
Bootstrap seed: 42
Bootstrap completed: True

Event headline metrics: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_event_headline_metrics.csv
Correction-versus-learning results: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_correction_vs_learning.csv

The main post-hoc narrative can now be drafted.
READY FOR NOTEBOOK 04: subgroup and tail analysis.


In [23]:
from __future__ import annotations

import json
import math

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from IPython.display import display


THESIS_DIR = Path(
    "/home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS"
)

CHAPTER4_ROOT = THESIS_DIR / "outputs" / "chapter4"

FINAL_DIR = CHAPTER4_ROOT / "posthoc_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)


INTEGRITY_JSON_PATH = (
    FINAL_DIR
    / "posthoc_integrity_audit.json"
)

EVENT_OUTCOME_PATH = (
    FINAL_DIR
    / "posthoc_event_paired_outcomes.parquet"
)

EVENT_HEADLINE_PATH = (
    FINAL_DIR
    / "posthoc_event_headline_metrics.csv"
)


TAIL_METRICS_PATH = (
    FINAL_DIR
    / "posthoc_event_tail_metrics.csv"
)

SUBGROUP_METRICS_PATH = (
    FINAL_DIR
    / "posthoc_subgroup_metrics.csv"
)

SUBGROUP_MACRO_PATH = (
    FINAL_DIR
    / "posthoc_subgroup_macro_summary.csv"
)

FOLD_METRICS_PATH = (
    FINAL_DIR
    / "posthoc_fold_metrics.csv"
)

FOLD_SUMMARY_PATH = (
    FINAL_DIR
    / "posthoc_fold_summary.csv"
)

WORST_SUBGROUPS_PATH = (
    FINAL_DIR
    / "posthoc_worst_subgroups.csv"
)

THESIS_READY_SUMMARY_PATH = (
    FINAL_DIR
    / "posthoc_thesis_ready_subgroup_summary.csv"
)

COMPLETION_PATH = (
    FINAL_DIR
    / "04_subgroups_and_tails_completion_summary.json"
)


EPSILON_PP = 1e-8


EXPERIMENT_ORDER = [
    "C90-1E",
    "L90-1E",
    "C110-1E",
    "L110-1E",
    "C90-2E",
    "L90-2E",
    "C110-2E",
    "L110-2E",
]


for required_path in [
    INTEGRITY_JSON_PATH,
    EVENT_OUTCOME_PATH,
    EVENT_HEADLINE_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


print("Event outcomes:", EVENT_OUTCOME_PATH)
print("Output directory:", FINAL_DIR)

Event outcomes: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_event_paired_outcomes.parquet
Output directory: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final


In [24]:
with INTEGRITY_JSON_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    integrity_payload = json.load(file)


if integrity_payload.get("overall_status") != "PASS":
    raise RuntimeError(
        "The mandatory integrity audit did not pass."
    )


event_outcomes = pd.read_parquet(
    EVENT_OUTCOME_PATH
)

event_headline = pd.read_csv(
    EVENT_HEADLINE_PATH
)


assert len(event_outcomes) == 39_736
assert set(
    event_outcomes["experiment_id"].unique()
) == set(EXPERIMENT_ORDER)


required_columns = {
    "experiment_id",
    "topology",
    "prior_view",
    "model_family",
    "sample_id",
    "fold",
    "y_true_pct",
    "y_pred_pct",
    "y_prior_pct",
    "model_abs_error_pp",
    "prior_abs_error_pp",
    "paired_gain_pp",
    "eps_outcome",
    "case",
    "broad_fault_family",
    "y_fault_line",
    "event_type",
    "location_bin_10pp",
}


missing_columns = sorted(
    required_columns
    - set(event_outcomes.columns)
)


if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )


print(
    "Physical-event rows:",
    f"{len(event_outcomes):,}",
)

display(
    event_outcomes[
        "experiment_id"
    ].value_counts().reindex(
        EXPERIMENT_ORDER
    ).rename("events")
)

Physical-event rows: 39,736


experiment_id
C90-1E     9022
L90-1E     9022
C110-1E     912
L110-1E     912
C90-2E     9022
L90-2E     9022
C110-2E     912
L110-2E     912
Name: events, dtype: int64

In [25]:
def finite_array(
    values: pd.Series,
) -> np.ndarray:
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    return numeric[
        np.isfinite(numeric)
    ]


def safe_relative_reduction(
    prior_value: float,
    model_value: float,
) -> float:
    if (
        not np.isfinite(prior_value)
        or abs(prior_value) <= EPSILON_PP
    ):
        return np.nan

    return float(
        100.0
        * (prior_value - model_value)
        / prior_value
    )


def cvar_upper(
    values: np.ndarray,
    quantile: float = 0.95,
) -> float:
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    threshold = np.quantile(
        values,
        quantile,
    )

    tail = values[
        values >= threshold
    ]

    if len(tail) == 0:
        return np.nan

    return float(
        np.mean(tail)
    )


def error_distribution_metrics(
    errors: np.ndarray,
    prefix: str,
) -> dict[str, float]:
    errors = np.asarray(
        errors,
        dtype=float,
    )

    errors = errors[
        np.isfinite(errors)
    ]

    if len(errors) == 0:
        return {
            f"{prefix}_mae_pp": np.nan,
            f"{prefix}_median_ae_pp": np.nan,
            f"{prefix}_p90_ae_pp": np.nan,
            f"{prefix}_p95_ae_pp": np.nan,
            f"{prefix}_p99_ae_pp": np.nan,
            f"{prefix}_max_ae_pp": np.nan,
            f"{prefix}_cvar95_pp": np.nan,
            f"{prefix}_exceed_5pp_rate_pct": np.nan,
            f"{prefix}_exceed_10pp_rate_pct": np.nan,
            f"{prefix}_exceed_20pp_rate_pct": np.nan,
            f"{prefix}_exceed_30pp_rate_pct": np.nan,
        }

    return {
        f"{prefix}_mae_pp": float(
            np.mean(errors)
        ),
        f"{prefix}_median_ae_pp": float(
            np.median(errors)
        ),
        f"{prefix}_p90_ae_pp": float(
            np.quantile(errors, 0.90)
        ),
        f"{prefix}_p95_ae_pp": float(
            np.quantile(errors, 0.95)
        ),
        f"{prefix}_p99_ae_pp": float(
            np.quantile(errors, 0.99)
        ),
        f"{prefix}_max_ae_pp": float(
            np.max(errors)
        ),
        f"{prefix}_cvar95_pp": (
            cvar_upper(errors, 0.95)
        ),
        f"{prefix}_exceed_5pp_rate_pct": float(
            100.0 * np.mean(errors > 5.0)
        ),
        f"{prefix}_exceed_10pp_rate_pct": float(
            100.0 * np.mean(errors > 10.0)
        ),
        f"{prefix}_exceed_20pp_rate_pct": float(
            100.0 * np.mean(errors > 20.0)
        ),
        f"{prefix}_exceed_30pp_rate_pct": float(
            100.0 * np.mean(errors > 30.0)
        ),
    }


def summarize_group(
    frame: pd.DataFrame,
) -> dict[str, Any]:
    model_error = finite_array(
        frame["model_abs_error_pp"]
    )

    prior_error = finite_array(
        frame["prior_abs_error_pp"]
    )

    gain = finite_array(
        frame["paired_gain_pp"]
    )

    if not (
        len(model_error)
        == len(prior_error)
        == len(gain)
        == len(frame)
    ):
        raise RuntimeError(
            "Non-finite values found in subgroup."
        )

    improved_count = int(
        np.sum(gain > EPSILON_PP)
    )

    worsened_count = int(
        np.sum(gain < -EPSILON_PP)
    )

    tied_count = int(
        len(gain)
        - improved_count
        - worsened_count
    )

    prior_mae = float(
        np.mean(prior_error)
    )

    model_mae = float(
        np.mean(model_error)
    )

    result = {
        "support": int(len(frame)),
        "prior_mae_pp": prior_mae,
        "model_mae_pp": model_mae,
        "mae_reduction_pp": (
            prior_mae - model_mae
        ),
        "relative_mae_reduction_pct": (
            safe_relative_reduction(
                prior_mae,
                model_mae,
            )
        ),
        "mean_paired_gain_pp": float(
            np.mean(gain)
        ),
        "median_paired_gain_pp": float(
            np.median(gain)
        ),
        "improved_count": improved_count,
        "worsened_count": worsened_count,
        "tied_count": tied_count,
        "improved_rate_pct": float(
            100.0
            * improved_count
            / len(gain)
        ),
        "worsened_rate_pct": float(
            100.0
            * worsened_count
            / len(gain)
        ),
        "tied_rate_pct": float(
            100.0
            * tied_count
            / len(gain)
        ),
        "improved_ge_5pp_rate_pct": float(
            100.0
            * np.mean(gain >= 5.0)
        ),
        "worsened_ge_5pp_rate_pct": float(
            100.0
            * np.mean(gain <= -5.0)
        ),
    }

    result.update(
        error_distribution_metrics(
            prior_error,
            "prior",
        )
    )

    result.update(
        error_distribution_metrics(
            model_error,
            "model",
        )
    )

    return result


def supported_threshold(
    topology: str,
    experiment_support: int,
) -> int:
    if topology == "90kv":
        return max(
            50,
            int(
                math.ceil(
                    0.01
                    * experiment_support
                )
            ),
        )

    return max(
        10,
        int(
            math.ceil(
                0.01
                * experiment_support
            )
        ),
    )

In [26]:
tail_rows = []


for experiment_id in EXPERIMENT_ORDER:
    frame = event_outcomes.loc[
        event_outcomes["experiment_id"]
        == experiment_id
    ].copy()

    summary = summarize_group(frame)

    tail_rows.append(
        {
            "experiment_id": experiment_id,
            "topology": frame[
                "topology"
            ].iloc[0],
            "prior_view": frame[
                "prior_view"
            ].iloc[0],
            "model_family": frame[
                "model_family"
            ].iloc[0],
            **summary,
        }
    )


tail_metrics = pd.DataFrame(
    tail_rows
)


experiment_order_map = {
    experiment_id: index
    for index, experiment_id
    in enumerate(EXPERIMENT_ORDER)
}


tail_metrics["_experiment_order"] = (
    tail_metrics["experiment_id"].map(
        experiment_order_map
    )
)

tail_metrics = (
    tail_metrics
    .sort_values(
        "_experiment_order",
        kind="stable",
    )
    .drop(columns="_experiment_order")
    .reset_index(drop=True)
)


tail_metrics.to_csv(
    TAIL_METRICS_PATH,
    index=False,
)


print("Saved:", TAIL_METRICS_PATH)

display(
    tail_metrics[
        [
            "experiment_id",
            "support",
            "prior_mae_pp",
            "model_mae_pp",
            "mae_reduction_pp",
            "model_p90_ae_pp",
            "model_p95_ae_pp",
            "model_p99_ae_pp",
            "model_cvar95_pp",
            "model_max_ae_pp",
            "model_exceed_10pp_rate_pct",
            "model_exceed_20pp_rate_pct",
        ]
    ]
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_event_tail_metrics.csv


,experiment_id,support,prior_mae_pp,model_mae_pp,mae_reduction_pp,model_p90_ae_pp,model_p95_ae_pp,model_p99_ae_pp,model_cvar95_pp,model_max_ae_pp,model_exceed_10pp_rate_pct,model_exceed_20pp_rate_pct
0,C90-1E,9022,7.322130,6.265098,1.057032,13.964525,20.365432,45.407452,35.836683,99.959648,17.335402,5.176236
1,L90-1E,9022,7.322130,6.288006,1.034124,13.507031,20.756001,50.703904,38.544771,99.959648,16.725781,5.331412
2,C110-1E,912,11.692447,6.636678,5.055769,19.227699,20.000000,50.000000,33.758650,80.000001,20.833333,7.565789
3,L110-1E,912,11.692447,7.825728,3.866719,19.999999,23.487724,50.000000,40.539749,80.000001,28.179825,9.978070
4,C90-2E,9022,1.148614,1.347025,-0.198410,1.770787,3.024826,26.944893,16.396556,98.767263,2.105963,1.341166
5,L90-2E,9022,1.148614,1.153115,-0.004500,1.379534,2.687269,27.296386,16.558445,98.767263,2.128131,1.385502
6,C110-2E,912,0.249126,0.468317,-0.219191,1.109753,1.466556,2.213075,1.930941,3.276178,0.000000,0.000000
7,L110-2E,912,0.249126,0.238907,0.010220,0.527329,0.683732,1.251428,1.030423,2.428544,0.000000,0.000000


In [27]:
SUBGROUP_DIMENSIONS = {
    "fault_case": "case",
    "broad_fault_family": (
        "broad_fault_family"
    ),
    "faulted_line": "y_fault_line",
    "event_type": "event_type",
    "location_bin_10pp": (
        "location_bin_10pp"
    ),
    "outer_fold": "fold",
}


subgroup_rows = []


for experiment_id in EXPERIMENT_ORDER:
    experiment_frame = event_outcomes.loc[
        event_outcomes["experiment_id"]
        == experiment_id
    ].copy()

    topology = str(
        experiment_frame["topology"].iloc[0]
    )

    prior_view = str(
        experiment_frame["prior_view"].iloc[0]
    )

    model_family = str(
        experiment_frame["model_family"].iloc[0]
    )

    experiment_support = len(
        experiment_frame
    )

    minimum_supported = (
        supported_threshold(
            topology,
            experiment_support,
        )
    )

    for dimension_name, column in (
        SUBGROUP_DIMENSIONS.items()
    ):
        working = experiment_frame.copy()

        working["_subgroup_value"] = (
            working[column]
            .astype("string")
            .fillna("<missing>")
        )

        for subgroup_value, subgroup in (
            working.groupby(
                "_subgroup_value",
                dropna=False,
                observed=False,
                sort=True,
            )
        ):
            summary = summarize_group(
                subgroup
            )

            subgroup_rows.append(
                {
                    "experiment_id": (
                        experiment_id
                    ),
                    "topology": topology,
                    "prior_view": prior_view,
                    "model_family": (
                        model_family
                    ),
                    "subgroup_dimension": (
                        dimension_name
                    ),
                    "subgroup_column": (
                        column
                    ),
                    "subgroup_value": str(
                        subgroup_value
                    ),
                    "minimum_supported": (
                        minimum_supported
                    ),
                    "supported_for_worst_group": (
                        summary["support"]
                        >= minimum_supported
                    ),
                    **summary,
                }
            )


subgroup_metrics = pd.DataFrame(
    subgroup_rows
)


subgroup_metrics.to_csv(
    SUBGROUP_METRICS_PATH,
    index=False,
)


print(
    "Subgroup result rows:",
    len(subgroup_metrics),
)

print("Saved:", SUBGROUP_METRICS_PATH)

display(
    subgroup_metrics.head(20)
)

Subgroup result rows: 284
Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_subgroup_metrics.csv


,experiment_id,topology,prior_view,model_family,subgroup_dimension,subgroup_column,subgroup_value,minimum_supported,supported_for_worst_group,support,...,model_median_ae_pp,model_p90_ae_pp,model_p95_ae_pp,model_p99_ae_pp,model_max_ae_pp,model_cvar95_pp,model_exceed_5pp_rate_pct,model_exceed_10pp_rate_pct,model_exceed_20pp_rate_pct,model_exceed_30pp_rate_pct
0,C90-1E,90kv,1E,correction,fault_case,case,3ph,91,True,2273,...,2.735458,11.135285,16.585880,37.646824,81.215347,29.969242,28.024637,11.702596,3.871535,1.803784
1,C90-1E,90kv,1E,correction,fault_case,case,ll_ab,91,True,725,...,4.545671,13.280901,20.277652,41.281374,94.534408,35.912639,46.758621,19.172414,5.241379,2.344828
2,C90-1E,90kv,1E,correction,fault_case,case,ll_bc,91,True,754,...,4.049771,12.127511,16.198074,33.510675,94.795958,30.115616,42.042440,16.180371,3.448276,1.591512
3,C90-1E,90kv,1E,correction,fault_case,case,ll_ca,91,True,772,...,4.410280,13.852170,21.527732,44.041814,91.738982,35.686919,45.725389,19.818653,5.310881,2.331606
4,C90-1E,90kv,1E,correction,fault_case,case,llg_ab,91,True,725,...,3.421772,14.266323,21.151830,46.038493,90.849149,37.564060,34.482759,16.413793,5.379310,2.482759
5,C90-1E,90kv,1E,correction,fault_case,case,llg_bc,91,True,735,...,3.084874,13.601991,18.896315,41.093814,99.959648,36.303781,32.517007,15.510204,4.761905,2.857143
6,C90-1E,90kv,1E,correction,fault_case,case,llg_ca,91,True,759,...,2.934358,14.218745,23.210502,49.590635,76.742116,39.205716,33.860343,15.810277,6.060606,3.293808
7,C90-1E,90kv,1E,correction,fault_case,case,slg_a,91,True,765,...,4.690194,16.494177,20.766234,53.821238,87.564260,36.628115,46.666667,24.444444,6.143791,2.352941
8,C90-1E,90kv,1E,correction,fault_case,case,slg_b,91,True,753,...,4.437625,17.539202,24.952805,57.717793,97.456992,45.569696,46.082337,22.974768,7.171315,3.585657
9,C90-1E,90kv,1E,correction,fault_case,case,slg_c,91,True,761,...,4.781893,16.388059,21.922812,53.117855,77.469496,37.567735,48.357424,22.470434,6.964520,2.628121


In [28]:
macro_rows = []


for (
    experiment_id,
    dimension,
), frame in subgroup_metrics.groupby(
    [
        "experiment_id",
        "subgroup_dimension",
    ],
    sort=False,
    observed=False,
):
    supported = frame.loc[
        frame[
            "supported_for_worst_group"
        ]
    ].copy()

    if supported.empty:
        supported = frame.copy()

    worst_model_row = supported.sort_values(
        [
            "model_mae_pp",
            "support",
        ],
        ascending=[
            False,
            False,
        ],
        kind="stable",
    ).iloc[0]

    worst_gain_row = supported.sort_values(
        [
            "mean_paired_gain_pp",
            "support",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    ).iloc[0]

    macro_rows.append(
        {
            "experiment_id": experiment_id,
            "subgroup_dimension": dimension,
            "subgroup_count": int(
                len(frame)
            ),
            "supported_subgroup_count": int(
                len(supported)
            ),
            "macro_prior_mae_pp": float(
                supported[
                    "prior_mae_pp"
                ].mean()
            ),
            "macro_model_mae_pp": float(
                supported[
                    "model_mae_pp"
                ].mean()
            ),
            "macro_mean_gain_pp": float(
                supported[
                    "mean_paired_gain_pp"
                ].mean()
            ),
            "macro_improved_rate_pct": float(
                supported[
                    "improved_rate_pct"
                ].mean()
            ),
            "worst_model_mae_value": (
                worst_model_row[
                    "subgroup_value"
                ]
            ),
            "worst_model_mae_pp": float(
                worst_model_row[
                    "model_mae_pp"
                ]
            ),
            "worst_gain_value": (
                worst_gain_row[
                    "subgroup_value"
                ]
            ),
            "worst_mean_gain_pp": float(
                worst_gain_row[
                    "mean_paired_gain_pp"
                ]
            ),
        }
    )


subgroup_macro = pd.DataFrame(
    macro_rows
)

subgroup_macro.to_csv(
    SUBGROUP_MACRO_PATH,
    index=False,
)


fold_metrics = subgroup_metrics.loc[
    subgroup_metrics[
        "subgroup_dimension"
    ] == "outer_fold"
].copy()


fold_metrics.to_csv(
    FOLD_METRICS_PATH,
    index=False,
)


fold_summary_rows = []


for experiment_id in EXPERIMENT_ORDER:
    frame = fold_metrics.loc[
        fold_metrics["experiment_id"]
        == experiment_id
    ].copy()

    worst_model_fold = frame.sort_values(
        "model_mae_pp",
        ascending=False,
        kind="stable",
    ).iloc[0]

    worst_gain_fold = frame.sort_values(
        "mean_paired_gain_pp",
        ascending=True,
        kind="stable",
    ).iloc[0]

    fold_summary_rows.append(
        {
            "experiment_id": experiment_id,
            "fold_count": int(
                len(frame)
            ),
            "fold_model_mae_mean_pp": float(
                frame["model_mae_pp"].mean()
            ),
            "fold_model_mae_std_pp": float(
                frame[
                    "model_mae_pp"
                ].std(ddof=0)
            ),
            "fold_model_mae_min_pp": float(
                frame["model_mae_pp"].min()
            ),
            "fold_model_mae_max_pp": float(
                frame["model_mae_pp"].max()
            ),
            "worst_fold_by_model_mae": (
                worst_model_fold[
                    "subgroup_value"
                ]
            ),
            "worst_fold_model_mae_pp": float(
                worst_model_fold[
                    "model_mae_pp"
                ]
            ),
            "worst_fold_by_gain": (
                worst_gain_fold[
                    "subgroup_value"
                ]
            ),
            "worst_fold_mean_gain_pp": float(
                worst_gain_fold[
                    "mean_paired_gain_pp"
                ]
            ),
            "fold_gain_mean_pp": float(
                frame[
                    "mean_paired_gain_pp"
                ].mean()
            ),
            "fold_gain_std_pp": float(
                frame[
                    "mean_paired_gain_pp"
                ].std(ddof=0)
            ),
            "folds_with_positive_gain": int(
                (
                    frame[
                        "mean_paired_gain_pp"
                    ] > 0.0
                ).sum()
            ),
            "folds_with_negative_gain": int(
                (
                    frame[
                        "mean_paired_gain_pp"
                    ] < 0.0
                ).sum()
            ),
        }
    )


fold_summary = pd.DataFrame(
    fold_summary_rows
)

fold_summary.to_csv(
    FOLD_SUMMARY_PATH,
    index=False,
)


print("Saved:", SUBGROUP_MACRO_PATH)
print("Saved:", FOLD_METRICS_PATH)
print("Saved:", FOLD_SUMMARY_PATH)

display(fold_summary)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_subgroup_macro_summary.csv
Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_fold_metrics.csv
Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_fold_summary.csv


,experiment_id,fold_count,fold_model_mae_mean_pp,fold_model_mae_std_pp,fold_model_mae_min_pp,fold_model_mae_max_pp,worst_fold_by_model_mae,worst_fold_model_mae_pp,worst_fold_by_gain,worst_fold_mean_gain_pp,fold_gain_mean_pp,fold_gain_std_pp,folds_with_positive_gain,folds_with_negative_gain
0,C90-1E,5,6.265099,0.223614,6.067628,6.652824,3,6.652824,2,0.849403,1.057034,0.136455,5,0
1,L90-1E,5,6.288018,0.209167,6.059611,6.658091,3,6.658091,1,0.965106,1.034114,0.054273,5,0
2,C110-1E,5,6.636596,0.821716,5.214521,7.334237,1,7.334237,1,4.407426,5.056082,0.523419,5,0
3,L110-1E,5,7.825447,0.987993,6.443070,9.422429,1,9.422429,1,2.319234,3.867231,1.176330,5,0
4,C90-2E,5,1.346996,0.246740,1.119880,1.827926,0,1.827926,0,-0.399704,-0.198394,0.108084,0,5
5,L90-2E,5,1.153103,0.155119,0.978828,1.432406,0,1.432406,3,-0.006929,-0.004500,0.002405,0,5
6,C110-2E,5,0.468325,0.077895,0.356812,0.572769,4,0.572769,4,-0.287554,-0.219240,0.073473,0,5
7,L110-2E,5,0.238859,0.020035,0.212783,0.270288,4,0.270288,0,-0.000167,0.010226,0.007766,4,1


In [29]:
worst_rows = []


for (
    experiment_id,
    dimension,
), frame in subgroup_metrics.groupby(
    [
        "experiment_id",
        "subgroup_dimension",
    ],
    sort=False,
    observed=False,
):
    supported = frame.loc[
        frame[
            "supported_for_worst_group"
        ]
    ].copy()

    if supported.empty:
        continue

    worst_model = supported.sort_values(
        [
            "model_mae_pp",
            "support",
        ],
        ascending=[
            False,
            False,
        ],
        kind="stable",
    ).iloc[0]

    worst_gain = supported.sort_values(
        [
            "mean_paired_gain_pp",
            "support",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    ).iloc[0]

    highest_worsening_rate = (
        supported.sort_values(
            [
                "worsened_rate_pct",
                "support",
            ],
            ascending=[
                False,
                False,
            ],
            kind="stable",
        ).iloc[0]
    )

    for criterion, row in [
        (
            "highest_model_mae",
            worst_model,
        ),
        (
            "lowest_mean_paired_gain",
            worst_gain,
        ),
        (
            "highest_worsened_rate",
            highest_worsening_rate,
        ),
    ]:
        worst_rows.append(
            {
                "experiment_id": (
                    experiment_id
                ),
                "topology": row[
                    "topology"
                ],
                "prior_view": row[
                    "prior_view"
                ],
                "model_family": row[
                    "model_family"
                ],
                "subgroup_dimension": (
                    dimension
                ),
                "selection_criterion": (
                    criterion
                ),
                "subgroup_value": row[
                    "subgroup_value"
                ],
                "support": int(
                    row["support"]
                ),
                "minimum_supported": int(
                    row[
                        "minimum_supported"
                    ]
                ),
                "prior_mae_pp": float(
                    row["prior_mae_pp"]
                ),
                "model_mae_pp": float(
                    row["model_mae_pp"]
                ),
                "mae_reduction_pp": float(
                    row[
                        "mae_reduction_pp"
                    ]
                ),
                "mean_paired_gain_pp": float(
                    row[
                        "mean_paired_gain_pp"
                    ]
                ),
                "improved_rate_pct": float(
                    row[
                        "improved_rate_pct"
                    ]
                ),
                "worsened_rate_pct": float(
                    row[
                        "worsened_rate_pct"
                    ]
                ),
                "model_p95_ae_pp": float(
                    row[
                        "model_p95_ae_pp"
                    ]
                ),
                "model_cvar95_pp": float(
                    row[
                        "model_cvar95_pp"
                    ]
                ),
            }
        )


worst_subgroups = pd.DataFrame(
    worst_rows
)

worst_subgroups.to_csv(
    WORST_SUBGROUPS_PATH,
    index=False,
)


print("Saved:", WORST_SUBGROUPS_PATH)

display(
    worst_subgroups.loc[
        worst_subgroups[
            "subgroup_dimension"
        ].isin(
            [
                "fault_case",
                "faulted_line",
                "location_bin_10pp",
            ]
        )
        & (
            worst_subgroups[
                "selection_criterion"
            ]
            == "lowest_mean_paired_gain"
        )
    ]
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_worst_subgroups.csv


,experiment_id,topology,prior_view,model_family,subgroup_dimension,selection_criterion,subgroup_value,support,minimum_supported,prior_mae_pp,model_mae_pp,mae_reduction_pp,mean_paired_gain_pp,improved_rate_pct,worsened_rate_pct,model_p95_ae_pp,model_cvar95_pp
1,C90-1E,90kv,1E,correction,fault_case,lowest_mean_paired_gain,ll_ca,772,91,6.878993,6.878993,0.000000,0.000000,0.000000,0.000000,21.527732,35.686919
7,C90-1E,90kv,1E,correction,faulted_line,lowest_mean_paired_gain,Line_2_3_b,1338,91,7.943855,6.936912,1.006944,1.006944,41.405082,31.016442,21.646382,39.460676
13,C90-1E,90kv,1E,correction,location_bin_10pp,lowest_mean_paired_gain,90-100,872,91,9.257652,9.453109,-0.195457,-0.195457,34.518349,34.403670,32.602120,52.388171
19,L90-1E,90kv,1E,combination,fault_case,lowest_mean_paired_gain,ll_ca,772,91,6.878993,6.878993,0.000000,0.000000,0.000000,0.000000,21.527732,35.686919
25,L90-1E,90kv,1E,combination,faulted_line,lowest_mean_paired_gain,Line_2_3_b,1338,91,7.943855,7.061456,0.882399,0.882399,44.917788,28.400598,24.771193,43.782860
31,L90-1E,90kv,1E,combination,location_bin_10pp,lowest_mean_paired_gain,90-100,872,91,9.257652,9.719421,-0.461769,-0.461769,34.633028,36.009174,32.281901,56.594418
37,C110-1E,110kv,1E,correction,fault_case,lowest_mean_paired_gain,ll_ab,60,10,4.886366,4.886366,0.000000,0.000000,0.000000,0.000000,12.478250,17.530326
43,C110-1E,110kv,1E,correction,faulted_line,lowest_mean_paired_gain,MainLn2-3B,228,10,10.338156,7.600238,2.737919,2.737919,32.894737,28.947368,30.971167,54.195695
49,C110-1E,110kv,1E,correction,location_bin_10pp,lowest_mean_paired_gain,50-60,192,10,12.443488,10.023259,2.420228,2.420228,37.500000,37.500000,50.000000,50.000000
55,L110-1E,110kv,1E,combination,fault_case,lowest_mean_paired_gain,ll_ab,60,10,4.886366,4.886366,0.000000,0.000000,0.000000,0.000000,12.478250,17.530326


In [30]:
thesis_ready_rows = []


for experiment_id in EXPERIMENT_ORDER:
    overall = tail_metrics.loc[
        tail_metrics["experiment_id"]
        == experiment_id
    ].iloc[0]

    case_worst = worst_subgroups.loc[
        (
            worst_subgroups[
                "experiment_id"
            ] == experiment_id
        )
        & (
            worst_subgroups[
                "subgroup_dimension"
            ] == "fault_case"
        )
        & (
            worst_subgroups[
                "selection_criterion"
            ]
            == "lowest_mean_paired_gain"
        )
    ]

    line_worst = worst_subgroups.loc[
        (
            worst_subgroups[
                "experiment_id"
            ] == experiment_id
        )
        & (
            worst_subgroups[
                "subgroup_dimension"
            ] == "faulted_line"
        )
        & (
            worst_subgroups[
                "selection_criterion"
            ]
            == "lowest_mean_paired_gain"
        )
    ]

    location_worst = worst_subgroups.loc[
        (
            worst_subgroups[
                "experiment_id"
            ] == experiment_id
        )
        & (
            worst_subgroups[
                "subgroup_dimension"
            ]
            == "location_bin_10pp"
        )
        & (
            worst_subgroups[
                "selection_criterion"
            ]
            == "lowest_mean_paired_gain"
        )
    ]

    fold_row = fold_summary.loc[
        fold_summary["experiment_id"]
        == experiment_id
    ].iloc[0]

    thesis_ready_rows.append(
        {
            "experiment_id": experiment_id,
            "prior_mae_pp": float(
                overall["prior_mae_pp"]
            ),
            "model_mae_pp": float(
                overall["model_mae_pp"]
            ),
            "mae_reduction_pp": float(
                overall[
                    "mae_reduction_pp"
                ]
            ),
            "model_p90_ae_pp": float(
                overall[
                    "model_p90_ae_pp"
                ]
            ),
            "model_p95_ae_pp": float(
                overall[
                    "model_p95_ae_pp"
                ]
            ),
            "model_p99_ae_pp": float(
                overall[
                    "model_p99_ae_pp"
                ]
            ),
            "model_cvar95_pp": float(
                overall[
                    "model_cvar95_pp"
                ]
            ),
            "model_exceed_10pp_rate_pct": float(
                overall[
                    "model_exceed_10pp_rate_pct"
                ]
            ),
            "worst_case": (
                case_worst.iloc[0][
                    "subgroup_value"
                ]
                if not case_worst.empty
                else None
            ),
            "worst_case_mean_gain_pp": (
                float(
                    case_worst.iloc[0][
                        "mean_paired_gain_pp"
                    ]
                )
                if not case_worst.empty
                else np.nan
            ),
            "worst_line": (
                line_worst.iloc[0][
                    "subgroup_value"
                ]
                if not line_worst.empty
                else None
            ),
            "worst_line_mean_gain_pp": (
                float(
                    line_worst.iloc[0][
                        "mean_paired_gain_pp"
                    ]
                )
                if not line_worst.empty
                else np.nan
            ),
            "worst_location_bin": (
                location_worst.iloc[0][
                    "subgroup_value"
                ]
                if not location_worst.empty
                else None
            ),
            "worst_location_mean_gain_pp": (
                float(
                    location_worst.iloc[0][
                        "mean_paired_gain_pp"
                    ]
                )
                if not location_worst.empty
                else np.nan
            ),
            "worst_fold": fold_row[
                "worst_fold_by_gain"
            ],
            "worst_fold_mean_gain_pp": float(
                fold_row[
                    "worst_fold_mean_gain_pp"
                ]
            ),
            "fold_model_mae_std_pp": float(
                fold_row[
                    "fold_model_mae_std_pp"
                ]
            ),
            "folds_with_positive_gain": int(
                fold_row[
                    "folds_with_positive_gain"
                ]
            ),
            "folds_with_negative_gain": int(
                fold_row[
                    "folds_with_negative_gain"
                ]
            ),
        }
    )


thesis_ready_summary = pd.DataFrame(
    thesis_ready_rows
)

thesis_ready_summary.to_csv(
    THESIS_READY_SUMMARY_PATH,
    index=False,
)


print("Saved:", THESIS_READY_SUMMARY_PATH)

print("\n")
print("=" * 120)
print("THESIS-READY TAIL AND WORST-SUBGROUP SUMMARY")
print("=" * 120)

print(
    thesis_ready_summary.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_thesis_ready_subgroup_summary.csv


THESIS-READY TAIL AND WORST-SUBGROUP SUMMARY
experiment_id  prior_mae_pp  model_mae_pp  mae_reduction_pp  model_p90_ae_pp  model_p95_ae_pp  model_p99_ae_pp  model_cvar95_pp  model_exceed_10pp_rate_pct worst_case  worst_case_mean_gain_pp worst_line  worst_line_mean_gain_pp worst_location_bin  worst_location_mean_gain_pp worst_fold  worst_fold_mean_gain_pp  fold_model_mae_std_pp  folds_with_positive_gain  folds_with_negative_gain
       C90-1E      7.322130      6.265098          1.057032        13.964525        20.365432        45.407452        35.836683                   17.335402      ll_ca                 0.000000 Line_2_3_b                 1.006944             90-100                    -0.195457          2                 0.849403               0.223614                         5                         0
       L90-1E      7.322130      6.288006          1

In [31]:
focused_dimensions = (
    subgroup_metrics.loc[
        subgroup_metrics[
            "subgroup_dimension"
        ].isin(
            [
                "fault_case",
                "faulted_line",
            ]
        )
    ]
    .copy()
)


for experiment_id in EXPERIMENT_ORDER:
    print("\n" + "=" * 100)
    print(experiment_id)
    print("=" * 100)

    experiment_groups = (
        focused_dimensions.loc[
            focused_dimensions[
                "experiment_id"
            ] == experiment_id
        ]
    )

    for dimension in [
        "fault_case",
        "faulted_line",
    ]:
        print(f"\n{dimension.upper()}")

        display_columns = [
            "subgroup_value",
            "support",
            "prior_mae_pp",
            "model_mae_pp",
            "mae_reduction_pp",
            "improved_rate_pct",
            "worsened_rate_pct",
            "model_p95_ae_pp",
        ]

        print(
            experiment_groups.loc[
                experiment_groups[
                    "subgroup_dimension"
                ] == dimension,
                display_columns,
            ]
            .sort_values(
                "mae_reduction_pp",
                ascending=False,
                kind="stable",
            )
            .to_string(
                index=False,
                float_format=lambda value: (
                    f"{value:.6f}"
                ),
            )
        )


C90-1E

FAULT_CASE
subgroup_value  support  prior_mae_pp  model_mae_pp  mae_reduction_pp  improved_rate_pct  worsened_rate_pct  model_p95_ae_pp
        llg_ca      759      8.057334      6.071158          1.986176          65.085639          32.674572        23.210502
         slg_a      765      9.198530      7.377459          1.821071          61.437908          36.601307        20.766234
         slg_b      753      9.635106      7.820030          1.815076          63.346614          33.864542        24.952805
         slg_c      761      9.322632      7.521081          1.801551          61.629435          36.005256        21.922812
        llg_ab      725      7.853622      6.152390          1.701231          65.379310          32.413793        21.151830
        llg_bc      735      7.038019      5.740709          1.297310          63.809524          34.557823        18.896315
           3ph     2273      5.699066      4.946199          0.752867          47.250330          51.8697

In [32]:
all_experiments_present = bool(
    set(
        tail_metrics[
            "experiment_id"
        ]
    )
    == set(EXPERIMENT_ORDER)
)


all_fold_summaries_present = bool(
    set(
        fold_summary[
            "experiment_id"
        ]
    )
    == set(EXPERIMENT_ORDER)
)


completion_summary = {
    "notebook": (
        "04_posthoc_subgroups_and_tails"
    ),
    "status": "PASS",
    "analysis_unit": "physical_event",
    "experiments": EXPERIMENT_ORDER,
    "overall_tail_metrics_complete": (
        all_experiments_present
    ),
    "subgroup_dimensions": list(
        SUBGROUP_DIMENSIONS.keys()
    ),
    "subgroup_result_rows": int(
        len(subgroup_metrics)
    ),
    "fold_summaries_complete": (
        all_fold_summaries_present
    ),
    "tail_metrics": str(
        TAIL_METRICS_PATH
    ),
    "subgroup_metrics": str(
        SUBGROUP_METRICS_PATH
    ),
    "subgroup_macro_summary": str(
        SUBGROUP_MACRO_PATH
    ),
    "fold_metrics": str(
        FOLD_METRICS_PATH
    ),
    "fold_summary": str(
        FOLD_SUMMARY_PATH
    ),
    "worst_subgroups": str(
        WORST_SUBGROUPS_PATH
    ),
    "thesis_ready_summary": str(
        THESIS_READY_SUMMARY_PATH
    ),
    "next_notebook": (
        "05_posthoc_110kv_window_stability"
    ),
}


with COMPLETION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        completion_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("\n")
print("=" * 80)
print("NOTEBOOK 04 COMPLETE")
print("=" * 80)

print("Status: PASS")

print(
    "Experiments analyzed:",
    len(EXPERIMENT_ORDER),
)

print(
    "Subgroup dimensions:",
    len(SUBGROUP_DIMENSIONS),
)

print(
    "Subgroup result rows:",
    len(subgroup_metrics),
)

print(
    "Tail metrics complete:",
    all_experiments_present,
)

print(
    "Fold summaries complete:",
    all_fold_summaries_present,
)

print(
    "\nThesis-ready summary:",
    THESIS_READY_SUMMARY_PATH,
)

print(
    "\nREADY FOR NOTEBOOK 05: "
    "110 kV window-stability analysis."
)



NOTEBOOK 04 COMPLETE
Status: PASS
Experiments analyzed: 8
Subgroup dimensions: 6
Subgroup result rows: 284
Tail metrics complete: True
Fold summaries complete: True

Thesis-ready summary: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_thesis_ready_subgroup_summary.csv

READY FOR NOTEBOOK 05: 110 kV window-stability analysis.


In [33]:
from __future__ import annotations

import json

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from IPython.display import display


THESIS_DIR = Path(
    "/home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS"
)

CHAPTER4_ROOT = THESIS_DIR / "outputs" / "chapter4"

FINAL_DIR = CHAPTER4_ROOT / "posthoc_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)


INTEGRITY_JSON_PATH = (
    FINAL_DIR
    / "posthoc_integrity_audit.json"
)

UNIFIED_WINDOW_PATH = (
    FINAL_DIR
    / "posthoc_unified_window_predictions.parquet"
)

UNIFIED_EVENT_PATH = (
    FINAL_DIR
    / "posthoc_unified_event_predictions.parquet"
)


EVENT_STABILITY_PATH = (
    FINAL_DIR
    / "posthoc_110kv_event_stability.parquet"
)

WINDOW_INDEX_METRICS_PATH = (
    FINAL_DIR
    / "posthoc_110kv_window_index_metrics.csv"
)

WINDOW_HEADLINE_PATH = (
    FINAL_DIR
    / "posthoc_110kv_window_headline_metrics.csv"
)

STABILITY_SUBGROUP_PATH = (
    FINAL_DIR
    / "posthoc_110kv_stability_subgroups.csv"
)

CORRECTION_LEARNING_STABILITY_PATH = (
    FINAL_DIR
    / "posthoc_110kv_correction_vs_learning_stability.csv"
)

WORST_UNSTABLE_EVENTS_PATH = (
    FINAL_DIR
    / "posthoc_110kv_worst_unstable_events.csv"
)

COMPLETION_PATH = (
    FINAL_DIR
    / "05_window_stability_completion_summary.json"
)


EXPERIMENT_ORDER = [
    "C110-1E",
    "L110-1E",
    "C110-2E",
    "L110-2E",
]


EXPECTED_WINDOWS = {
    8,
    9,
    10,
    11,
}


NUMERIC_TOLERANCE = 1e-8


for required_path in [
    INTEGRITY_JSON_PATH,
    UNIFIED_WINDOW_PATH,
    UNIFIED_EVENT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


print("Unified window input:", UNIFIED_WINDOW_PATH)
print("Unified event input:", UNIFIED_EVENT_PATH)
print("Output directory:", FINAL_DIR)

Unified window input: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_window_predictions.parquet
Unified event input: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_unified_event_predictions.parquet
Output directory: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final


In [34]:
with INTEGRITY_JSON_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    integrity_payload = json.load(file)


if integrity_payload.get("overall_status") != "PASS":
    raise RuntimeError(
        "The mandatory integrity audit did not pass."
    )


unified_window = pd.read_parquet(
    UNIFIED_WINDOW_PATH
)

unified_event = pd.read_parquet(
    UNIFIED_EVENT_PATH
)


window_110 = unified_window.loc[
    unified_window["experiment_id"].isin(
        EXPERIMENT_ORDER
    )
].copy()

event_110 = unified_event.loc[
    unified_event["experiment_id"].isin(
        EXPERIMENT_ORDER
    )
].copy()


assert len(window_110) == 14_592
assert len(event_110) == 3_648


window_counts = (
    window_110["experiment_id"]
    .value_counts()
    .reindex(EXPERIMENT_ORDER)
)

event_counts = (
    event_110["experiment_id"]
    .value_counts()
    .reindex(EXPERIMENT_ORDER)
)


assert (window_counts == 3648).all()
assert (event_counts == 912).all()


observed_windows = set(
    pd.to_numeric(
        window_110["window_idx"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .unique()
)


assert observed_windows == EXPECTED_WINDOWS


print("110 kV window rows:", f"{len(window_110):,}")
print("110 kV event rows:", f"{len(event_110):,}")
print("Window indices:", sorted(observed_windows))

display(
    pd.DataFrame(
        {
            "window_rows": window_counts,
            "event_rows": event_counts,
        }
    )
)

110 kV window rows: 14,592
110 kV event rows: 3,648
Window indices: [np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


,window_rows,event_rows
experiment_id,,
C110-1E,3648,912
L110-1E,3648,912
C110-2E,3648,912
L110-2E,3648,912


In [35]:
def finite_array(
    values: pd.Series,
) -> np.ndarray:
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    return numeric[
        np.isfinite(numeric)
    ]


def std_ddof_zero(
    values: pd.Series,
) -> float:
    numeric = finite_array(values)

    if len(numeric) == 0:
        return np.nan

    return float(
        np.std(
            numeric,
            ddof=0,
        )
    )


def numeric_range(
    values: pd.Series,
) -> float:
    numeric = finite_array(values)

    if len(numeric) == 0:
        return np.nan

    return float(
        np.max(numeric)
        - np.min(numeric)
    )


def cvar_upper(
    values: np.ndarray,
    quantile: float = 0.95,
) -> float:
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    threshold = np.quantile(
        values,
        quantile,
    )

    tail = values[
        values >= threshold
    ]

    return float(
        np.mean(tail)
    )


def maximum_difference(
    left: pd.Series,
    right: pd.Series,
) -> float:
    left_numeric = pd.to_numeric(
        left,
        errors="coerce",
    )

    right_numeric = pd.to_numeric(
        right,
        errors="coerce",
    )

    valid = (
        np.isfinite(left_numeric)
        & np.isfinite(right_numeric)
    )

    if not valid.any():
        return np.inf

    return float(
        (
            left_numeric[valid]
            - right_numeric[valid]
        ).abs().max()
    )


def summarize_stability_group(
    frame: pd.DataFrame,
) -> dict[str, Any]:
    event_error = finite_array(
        frame["event_mean_abs_error_pp"]
    )

    window_std = finite_array(
        frame["window_prediction_std_pp"]
    )

    window_range = finite_array(
        frame["window_prediction_range_pp"]
    )

    averaging_gain = finite_array(
        frame["averaging_gain_pp"]
    )

    mean_vs_median_gain = finite_array(
        frame["mean_vs_median_gain_pp"]
    )

    if not (
        len(event_error)
        == len(window_std)
        == len(window_range)
        == len(averaging_gain)
        == len(mean_vs_median_gain)
        == len(frame)
    ):
        raise RuntimeError(
            "Non-finite values found in stability group."
        )

    return {
        "support": int(len(frame)),

        "event_mean_mae_pp": float(
            np.mean(event_error)
        ),

        "mean_window_mae_pp": float(
            frame[
                "mean_window_abs_error_pp"
            ].mean()
        ),

        "median_prediction_mae_pp": float(
            frame[
                "median_prediction_abs_error_pp"
            ].mean()
        ),

        "best_window_mae_pp": float(
            frame[
                "best_window_abs_error_pp"
            ].mean()
        ),

        "worst_window_mae_pp": float(
            frame[
                "worst_window_abs_error_pp"
            ].mean()
        ),

        "mean_window_prediction_std_pp": float(
            np.mean(window_std)
        ),

        "median_window_prediction_std_pp": float(
            np.median(window_std)
        ),

        "p95_window_prediction_std_pp": float(
            np.quantile(
                window_std,
                0.95,
            )
        ),

        "mean_window_prediction_range_pp": float(
            np.mean(window_range)
        ),

        "median_window_prediction_range_pp": float(
            np.median(window_range)
        ),

        "p95_window_prediction_range_pp": float(
            np.quantile(
                window_range,
                0.95,
            )
        ),

        "max_window_prediction_range_pp": float(
            np.max(window_range)
        ),

        "mean_averaging_gain_pp": float(
            np.mean(averaging_gain)
        ),

        "median_averaging_gain_pp": float(
            np.median(averaging_gain)
        ),

        "averaging_helped_rate_pct": float(
            100.0
            * np.mean(
                averaging_gain
                > NUMERIC_TOLERANCE
            )
        ),

        "averaging_hurt_rate_pct": float(
            100.0
            * np.mean(
                averaging_gain
                < -NUMERIC_TOLERANCE
            )
        ),

        "averaging_tied_rate_pct": float(
            100.0
            * np.mean(
                np.abs(
                    averaging_gain
                )
                <= NUMERIC_TOLERANCE
            )
        ),

        "mean_vs_median_gain_pp": float(
            np.mean(
                mean_vs_median_gain
            )
        ),

        "mean_better_than_median_rate_pct": float(
            100.0
            * np.mean(
                mean_vs_median_gain
                > NUMERIC_TOLERANCE
            )
        ),

        "median_better_than_mean_rate_pct": float(
            100.0
            * np.mean(
                mean_vs_median_gain
                < -NUMERIC_TOLERANCE
            )
        ),

        "event_error_p90_pp": float(
            np.quantile(
                event_error,
                0.90,
            )
        ),

        "event_error_p95_pp": float(
            np.quantile(
                event_error,
                0.95,
            )
        ),

        "event_error_p99_pp": float(
            np.quantile(
                event_error,
                0.99,
            )
        ),

        "event_error_cvar95_pp": (
            cvar_upper(
                event_error,
                0.95,
            )
        ),

        "spearman_std_vs_event_error": float(
            frame[
                "window_prediction_std_pp"
            ].corr(
                frame[
                    "event_mean_abs_error_pp"
                ],
                method="spearman",
            )
        ),

        "spearman_range_vs_event_error": float(
            frame[
                "window_prediction_range_pp"
            ].corr(
                frame[
                    "event_mean_abs_error_pp"
                ],
                method="spearman",
            )
        ),

        "spearman_range_vs_averaging_gain": float(
            frame[
                "window_prediction_range_pp"
            ].corr(
                frame[
                    "averaging_gain_pp"
                ],
                method="spearman",
            )
        ),
    }

In [36]:
event_stability_frames = []
verification_rows = []


for experiment_id in EXPERIMENT_ORDER:
    window_frame = window_110.loc[
        window_110["experiment_id"]
        == experiment_id
    ].copy()

    saved_event = event_110.loc[
        event_110["experiment_id"]
        == experiment_id
    ].copy()

    grouped = window_frame.groupby(
        "sample_id",
        sort=False,
        dropna=False,
    )

    stability = grouped.agg(
        experiment_id=(
            "experiment_id",
            "first",
        ),
        topology=(
            "topology",
            "first",
        ),
        prior_view=(
            "prior_view",
            "first",
        ),
        model_family=(
            "model_family",
            "first",
        ),
        fold=(
            "fold",
            "first",
        ),
        y_true_pct=(
            "y_true_pct",
            "first",
        ),
        y_prior_pct=(
            "y_prior_pct",
            "mean",
        ),
        event_mean_prediction_pct=(
            "y_pred_pct",
            "mean",
        ),
        event_median_prediction_pct=(
            "y_pred_pct",
            "median",
        ),
        minimum_window_prediction_pct=(
            "y_pred_pct",
            "min",
        ),
        maximum_window_prediction_pct=(
            "y_pred_pct",
            "max",
        ),
        window_prediction_std_pp=(
            "y_pred_pct",
            std_ddof_zero,
        ),
        window_prediction_range_pp=(
            "y_pred_pct",
            numeric_range,
        ),
        mean_window_abs_error_pp=(
            "model_abs_error_pp",
            "mean",
        ),
        best_window_abs_error_pp=(
            "model_abs_error_pp",
            "min",
        ),
        worst_window_abs_error_pp=(
            "model_abs_error_pp",
            "max",
        ),
        window_error_std_pp=(
            "model_abs_error_pp",
            std_ddof_zero,
        ),
        prior_window_error_mean_pp=(
            "prior_abs_error_pp",
            "mean",
        ),
        case=(
            "case",
            "first",
        ),
        broad_fault_family=(
            "broad_fault_family",
            "first",
        ),
        y_fault_line=(
            "y_fault_line",
            "first",
        ),
        event_type=(
            "event_type",
            "first",
        ),
        location_bin_10pp=(
            "location_bin_10pp",
            "first",
        ),
        window_rows=(
            "sample_id",
            "size",
        ),
        window_idx_count=(
            "window_idx",
            "nunique",
        ),
        fold_count=(
            "fold",
            "nunique",
        ),
        target_count=(
            "y_true_pct",
            "nunique",
        ),
    ).reset_index()

    stability[
        "event_mean_abs_error_pp"
    ] = (
        stability[
            "event_mean_prediction_pct"
        ]
        - stability["y_true_pct"]
    ).abs()

    stability[
        "median_prediction_abs_error_pp"
    ] = (
        stability[
            "event_median_prediction_pct"
        ]
        - stability["y_true_pct"]
    ).abs()

    stability[
        "prior_event_abs_error_pp"
    ] = (
        stability["y_prior_pct"]
        - stability["y_true_pct"]
    ).abs()

    stability[
        "averaging_gain_pp"
    ] = (
        stability[
            "mean_window_abs_error_pp"
        ]
        - stability[
            "event_mean_abs_error_pp"
        ]
    )

    stability[
        "mean_vs_median_gain_pp"
    ] = (
        stability[
            "median_prediction_abs_error_pp"
        ]
        - stability[
            "event_mean_abs_error_pp"
        ]
    )

    stability[
        "event_model_gain_over_prior_pp"
    ] = (
        stability[
            "prior_event_abs_error_pp"
        ]
        - stability[
            "event_mean_abs_error_pp"
        ]
    )

    stability[
        "oracle_window_gain_over_mean_pp"
    ] = (
        stability[
            "event_mean_abs_error_pp"
        ]
        - stability[
            "best_window_abs_error_pp"
        ]
    )

    stability[
        "mean_gain_over_worst_window_pp"
    ] = (
        stability[
            "worst_window_abs_error_pp"
        ]
        - stability[
            "event_mean_abs_error_pp"
        ]
    )

    stability[
        "experiment_event_key"
    ] = (
        stability["experiment_id"]
        .astype(str)
        + "::"
        + stability["sample_id"]
        .astype(str)
    )

    invalid_structure = stability.loc[
        (stability["window_rows"] != 4)
        | (
            stability["window_idx_count"]
            != 4
        )
        | (
            stability["fold_count"]
            != 1
        )
        | (
            stability["target_count"]
            != 1
        )
    ]

    if not invalid_structure.empty:
        raise RuntimeError(
            f"Invalid four-window event structure in "
            f"{experiment_id}."
        )

    comparison = stability.merge(
        saved_event[
            [
                "sample_id",
                "y_pred_pct",
                "model_abs_error_pp",
                "y_pred_std_pct",
                "y_pred_range_pct",
                "event_averaging_gain_pp",
            ]
        ],
        on="sample_id",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_saved",
        ),
    )

    prediction_difference = maximum_difference(
        comparison[
            "event_mean_prediction_pct"
        ],
        comparison["y_pred_pct"],
    )

    error_difference = maximum_difference(
        comparison[
            "event_mean_abs_error_pp"
        ],
        comparison[
            "model_abs_error_pp"
        ],
    )

    std_difference = maximum_difference(
        comparison[
            "window_prediction_std_pp"
        ],
        comparison[
            "y_pred_std_pct"
        ],
    )

    range_difference = maximum_difference(
        comparison[
            "window_prediction_range_pp"
        ],
        comparison[
            "y_pred_range_pct"
        ],
    )

    averaging_gain_difference = (
        maximum_difference(
            comparison[
                "averaging_gain_pp"
            ],
            comparison[
                "event_averaging_gain_pp"
            ],
        )
    )

    verification_ok = bool(
        len(comparison) == 912
        and prediction_difference
        <= NUMERIC_TOLERANCE
        and error_difference
        <= NUMERIC_TOLERANCE
        and std_difference
        <= NUMERIC_TOLERANCE
        and range_difference
        <= NUMERIC_TOLERANCE
        and averaging_gain_difference
        <= NUMERIC_TOLERANCE
    )

    verification_rows.append(
        {
            "experiment_id": experiment_id,
            "events": len(stability),
            "prediction_max_difference": (
                prediction_difference
            ),
            "error_max_difference": (
                error_difference
            ),
            "std_max_difference": (
                std_difference
            ),
            "range_max_difference": (
                range_difference
            ),
            "averaging_gain_max_difference": (
                averaging_gain_difference
            ),
            "verification_ok": verification_ok,
        }
    )

    event_stability_frames.append(
        stability
    )


event_stability = pd.concat(
    event_stability_frames,
    ignore_index=True,
)


stability_verification = pd.DataFrame(
    verification_rows
)


display(stability_verification)


if not stability_verification[
    "verification_ok"
].all():
    raise RuntimeError(
        "Reconstructed 110 kV stability values do not "
        "match the unified event table."
    )


event_stability.to_parquet(
    EVENT_STABILITY_PATH,
    index=False,
)


print("Saved:", EVENT_STABILITY_PATH)
print("Event-stability rows:", len(event_stability))

,experiment_id,events,prediction_max_difference,error_max_difference,std_max_difference,range_max_difference,averaging_gain_max_difference,verification_ok
0,C110-1E,912,0.0,0.0,0.0,0.0,0.0,True
1,L110-1E,912,0.0,0.0,0.0,0.0,0.0,True
2,C110-2E,912,0.0,0.0,0.0,0.0,0.0,True
3,L110-2E,912,0.0,0.0,0.0,0.0,0.0,True


Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_event_stability.parquet
Event-stability rows: 3648


In [37]:
window_index_rows = []


for experiment_id in EXPERIMENT_ORDER:
    experiment_frame = window_110.loc[
        window_110["experiment_id"]
        == experiment_id
    ].copy()

    for window_idx, subgroup in (
        experiment_frame.groupby(
            "window_idx",
            sort=True,
            observed=False,
        )
    ):
        model_error = finite_array(
            subgroup["model_abs_error_pp"]
        )

        prior_error = finite_array(
            subgroup["prior_abs_error_pp"]
        )

        paired_gain = (
            prior_error
            - model_error
        )

        window_index_rows.append(
            {
                "experiment_id": (
                    experiment_id
                ),
                "topology": "110kv",
                "prior_view": subgroup[
                    "prior_view"
                ].iloc[0],
                "model_family": subgroup[
                    "model_family"
                ].iloc[0],
                "window_idx": int(
                    window_idx
                ),
                "support": int(
                    len(subgroup)
                ),
                "prior_mae_pp": float(
                    np.mean(prior_error)
                ),
                "model_mae_pp": float(
                    np.mean(model_error)
                ),
                "mae_reduction_pp": float(
                    np.mean(paired_gain)
                ),
                "model_median_ae_pp": float(
                    np.median(model_error)
                ),
                "model_p90_ae_pp": float(
                    np.quantile(
                        model_error,
                        0.90,
                    )
                ),
                "model_p95_ae_pp": float(
                    np.quantile(
                        model_error,
                        0.95,
                    )
                ),
                "model_p99_ae_pp": float(
                    np.quantile(
                        model_error,
                        0.99,
                    )
                ),
                "model_cvar95_pp": (
                    cvar_upper(
                        model_error,
                        0.95,
                    )
                ),
                "improved_rate_pct": float(
                    100.0
                    * np.mean(
                        paired_gain
                        > NUMERIC_TOLERANCE
                    )
                ),
                "worsened_rate_pct": float(
                    100.0
                    * np.mean(
                        paired_gain
                        < -NUMERIC_TOLERANCE
                    )
                ),
            }
        )


window_index_metrics = pd.DataFrame(
    window_index_rows
)


window_index_metrics.to_csv(
    WINDOW_INDEX_METRICS_PATH,
    index=False,
)


print("Saved:", WINDOW_INDEX_METRICS_PATH)

display(window_index_metrics)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_window_index_metrics.csv


,experiment_id,topology,prior_view,model_family,window_idx,support,prior_mae_pp,model_mae_pp,mae_reduction_pp,model_median_ae_pp,model_p90_ae_pp,model_p95_ae_pp,model_p99_ae_pp,model_cvar95_pp,improved_rate_pct,worsened_rate_pct
0,C110-1E,110kv,1E,correction,8,912,11.692447,6.650962,5.041485,2.817249,19.281229,20.000000,50.000000,34.121550,43.530702,22.368421
1,C110-1E,110kv,1E,correction,9,912,11.692447,6.665891,5.026556,2.765366,19.231149,20.000000,50.000000,33.986173,42.214912,23.464912
2,C110-1E,110kv,1E,correction,10,912,11.692447,6.717914,4.974534,2.811025,19.999999,20.000000,50.000000,33.731241,42.434211,23.464912
3,C110-1E,110kv,1E,correction,11,912,11.692447,6.687507,5.004940,2.782689,19.766966,20.000000,50.000000,34.102769,42.982456,22.807018
4,L110-1E,110kv,1E,combination,8,912,11.692447,8.014934,3.677513,4.155976,20.000000,25.125048,50.000000,40.742021,46.710526,23.464912
5,L110-1E,110kv,1E,combination,9,912,11.692447,8.032152,3.660295,4.342815,20.000000,23.724699,50.000000,41.326377,44.736842,25.109649
6,L110-1E,110kv,1E,combination,10,912,11.692447,7.980033,3.712414,4.259676,20.000000,25.152313,50.000000,40.977700,46.710526,23.793860
7,L110-1E,110kv,1E,combination,11,912,11.692447,7.846939,3.845509,4.102919,19.999999,24.003298,50.000000,41.750812,46.271930,23.903509
8,C110-2E,110kv,2E,correction,8,912,0.952859,1.194311,-0.241453,0.999999,2.721226,3.314503,4.802655,4.308659,32.456140,58.442982
9,C110-2E,110kv,2E,correction,9,912,0.593107,0.853550,-0.260444,0.564367,2.075945,2.683445,4.434546,3.697555,32.894737,65.021930


In [38]:
headline_rows = []


for experiment_id in EXPERIMENT_ORDER:
    frame = event_stability.loc[
        event_stability["experiment_id"]
        == experiment_id
    ].copy()

    summary = summarize_stability_group(
        frame
    )

    window_idx_frame = (
        window_index_metrics.loc[
            window_index_metrics[
                "experiment_id"
            ] == experiment_id
        ]
    )

    best_window_row = (
        window_idx_frame.sort_values(
            "model_mae_pp",
            ascending=True,
            kind="stable",
        ).iloc[0]
    )

    worst_window_row = (
        window_idx_frame.sort_values(
            "model_mae_pp",
            ascending=False,
            kind="stable",
        ).iloc[0]
    )

    headline_rows.append(
        {
            "experiment_id": experiment_id,
            "prior_view": frame[
                "prior_view"
            ].iloc[0],
            "model_family": frame[
                "model_family"
            ].iloc[0],
            **summary,
            "best_individual_window_idx": int(
                best_window_row[
                    "window_idx"
                ]
            ),
            "best_individual_window_mae_pp": float(
                best_window_row[
                    "model_mae_pp"
                ]
            ),
            "worst_individual_window_idx": int(
                worst_window_row[
                    "window_idx"
                ]
            ),
            "worst_individual_window_mae_pp": float(
                worst_window_row[
                    "model_mae_pp"
                ]
            ),
            "individual_window_mae_spread_pp": float(
                worst_window_row[
                    "model_mae_pp"
                ]
                - best_window_row[
                    "model_mae_pp"
                ]
            ),
        }
    )


window_headline = pd.DataFrame(
    headline_rows
)


experiment_order_map = {
    experiment_id: index
    for index, experiment_id
    in enumerate(EXPERIMENT_ORDER)
}


window_headline[
    "_experiment_order"
] = window_headline[
    "experiment_id"
].map(
    experiment_order_map
)


window_headline = (
    window_headline
    .sort_values(
        "_experiment_order",
        kind="stable",
    )
    .drop(columns="_experiment_order")
    .reset_index(drop=True)
)


window_headline.to_csv(
    WINDOW_HEADLINE_PATH,
    index=False,
)


print("Saved:", WINDOW_HEADLINE_PATH)

display(
    window_headline[
        [
            "experiment_id",
            "support",
            "event_mean_mae_pp",
            "mean_window_mae_pp",
            "mean_averaging_gain_pp",
            "averaging_helped_rate_pct",
            "averaging_hurt_rate_pct",
            "mean_window_prediction_std_pp",
            "p95_window_prediction_range_pp",
            "best_individual_window_idx",
            "best_individual_window_mae_pp",
            "worst_individual_window_idx",
            "worst_individual_window_mae_pp",
            "spearman_range_vs_event_error",
        ]
    ]
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_window_headline_metrics.csv


,experiment_id,support,event_mean_mae_pp,mean_window_mae_pp,mean_averaging_gain_pp,averaging_helped_rate_pct,averaging_hurt_rate_pct,mean_window_prediction_std_pp,p95_window_prediction_range_pp,best_individual_window_idx,best_individual_window_mae_pp,worst_individual_window_idx,worst_individual_window_mae_pp,spearman_range_vs_event_error
0,C110-1E,912,6.636678,6.680568,0.043890,9.429825,0.0,0.478937,4.842148,8,6.650962,10,6.717914,0.134324
1,L110-1E,912,7.825728,7.968515,0.142787,15.460526,0.0,1.215045,11.905118,11,7.846939,9,8.032152,0.251399
2,C110-2E,912,0.468317,0.844704,0.376387,84.978070,0.0,0.841788,4.962128,10,0.653619,8,1.194311,0.425451
3,L110-2E,912,0.238907,0.498886,0.259979,85.526316,0.0,0.564757,3.804575,11,0.297605,8,0.862948,0.654166


In [39]:
STABILITY_DIMENSIONS = {
    "fault_case": "case",
    "faulted_line": "y_fault_line",
    "broad_fault_family": (
        "broad_fault_family"
    ),
}


subgroup_rows = []


for experiment_id in EXPERIMENT_ORDER:
    experiment_frame = event_stability.loc[
        event_stability["experiment_id"]
        == experiment_id
    ].copy()

    for dimension_name, column in (
        STABILITY_DIMENSIONS.items()
    ):
        working = experiment_frame.copy()

        working["_subgroup_value"] = (
            working[column]
            .astype("string")
            .fillna("<missing>")
        )

        for subgroup_value, subgroup in (
            working.groupby(
                "_subgroup_value",
                sort=True,
                observed=False,
            )
        ):
            summary = summarize_stability_group(
                subgroup
            )

            subgroup_rows.append(
                {
                    "experiment_id": (
                        experiment_id
                    ),
                    "prior_view": subgroup[
                        "prior_view"
                    ].iloc[0],
                    "model_family": subgroup[
                        "model_family"
                    ].iloc[0],
                    "subgroup_dimension": (
                        dimension_name
                    ),
                    "subgroup_value": str(
                        subgroup_value
                    ),
                    **summary,
                }
            )


stability_subgroups = pd.DataFrame(
    subgroup_rows
)


stability_subgroups.to_csv(
    STABILITY_SUBGROUP_PATH,
    index=False,
)


print("Saved:", STABILITY_SUBGROUP_PATH)
print("Stability subgroup rows:", len(stability_subgroups))


worst_case_stability = (
    stability_subgroups.loc[
        stability_subgroups[
            "subgroup_dimension"
        ] == "fault_case"
    ]
    .sort_values(
        [
            "experiment_id",
            "mean_window_prediction_range_pp",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    )
    .groupby(
        "experiment_id",
        sort=False,
    )
    .head(1)
)


display(
    worst_case_stability[
        [
            "experiment_id",
            "subgroup_value",
            "support",
            "mean_window_prediction_std_pp",
            "mean_window_prediction_range_pp",
            "mean_averaging_gain_pp",
            "averaging_helped_rate_pct",
            "event_mean_mae_pp",
        ]
    ]
)

/home/hpc/iwi5/iwi5305h/miniconda3/envs/Masters_thesis_env/lib/python3.14/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/home/hpc/iwi5/iwi5305h/miniconda3/envs/Masters_thesis_env/lib/python3.14/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/home/hpc/iwi5/iwi5305h/miniconda3/envs/Masters_thesis_env/lib/python3.14/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/home/hpc/iwi5/iwi5305h/miniconda3/envs/Masters_thesis_env/lib/python3.14/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_stability_subgroups.csv
Stability subgroup rows: 72


,experiment_id,subgroup_value,support,mean_window_prediction_std_pp,mean_window_prediction_range_pp,mean_averaging_gain_pp,averaging_helped_rate_pct,event_mean_mae_pp
9,C110-1E,slg_c,124,0.880555,2.259191,0.032475,8.064516,7.673769
44,C110-2E,slg_b,124,1.313770,3.280824,0.683269,88.709677,0.618966
27,L110-1E,slg_c,124,2.647199,6.808977,0.225294,20.161290,10.762084
62,L110-2E,slg_b,124,1.130365,2.779468,0.614762,94.354839,0.442772


In [40]:
PAIR_DEFINITIONS = [
    {
        "prior_view": "1E",
        "correction": "C110-1E",
        "combination": "L110-1E",
    },
    {
        "prior_view": "2E",
        "correction": "C110-2E",
        "combination": "L110-2E",
    },
]


comparison_rows = []


for pair in PAIR_DEFINITIONS:
    correction = event_stability.loc[
        event_stability["experiment_id"]
        == pair["correction"]
    ].copy()

    combination = event_stability.loc[
        event_stability["experiment_id"]
        == pair["combination"]
    ].copy()

    merged = correction.merge(
        combination,
        on="sample_id",
        how="inner",
        suffixes=(
            "_correction",
            "_combination",
        ),
        validate="one_to_one",
    )

    assert len(merged) == 912

    range_gain = (
        merged[
            "window_prediction_range_pp_correction"
        ]
        - merged[
            "window_prediction_range_pp_combination"
        ]
    )

    std_gain = (
        merged[
            "window_prediction_std_pp_correction"
        ]
        - merged[
            "window_prediction_std_pp_combination"
        ]
    )

    event_error_gain = (
        merged[
            "event_mean_abs_error_pp_correction"
        ]
        - merged[
            "event_mean_abs_error_pp_combination"
        ]
    )

    averaging_gain_difference = (
        merged[
            "averaging_gain_pp_combination"
        ]
        - merged[
            "averaging_gain_pp_correction"
        ]
    )

    comparison_rows.append(
        {
            "topology": "110kv",
            "prior_view": pair[
                "prior_view"
            ],
            "correction_experiment": (
                pair["correction"]
            ),
            "combination_experiment": (
                pair["combination"]
            ),
            "n_events": len(merged),

            "correction_mean_window_std_pp": float(
                merged[
                    "window_prediction_std_pp_correction"
                ].mean()
            ),

            "combination_mean_window_std_pp": float(
                merged[
                    "window_prediction_std_pp_combination"
                ].mean()
            ),

            "combination_std_reduction_pp": float(
                std_gain.mean()
            ),

            "combination_more_stable_std_rate_pct": float(
                100.0
                * np.mean(
                    std_gain
                    > NUMERIC_TOLERANCE
                )
            ),

            "correction_mean_window_range_pp": float(
                merged[
                    "window_prediction_range_pp_correction"
                ].mean()
            ),

            "combination_mean_window_range_pp": float(
                merged[
                    "window_prediction_range_pp_combination"
                ].mean()
            ),

            "combination_range_reduction_pp": float(
                range_gain.mean()
            ),

            "combination_more_stable_range_rate_pct": float(
                100.0
                * np.mean(
                    range_gain
                    > NUMERIC_TOLERANCE
                )
            ),

            "correction_event_mae_pp": float(
                merged[
                    "event_mean_abs_error_pp_correction"
                ].mean()
            ),

            "combination_event_mae_pp": float(
                merged[
                    "event_mean_abs_error_pp_combination"
                ].mean()
            ),

            "combination_event_mae_gain_pp": float(
                event_error_gain.mean()
            ),

            "combination_lower_event_error_rate_pct": float(
                100.0
                * np.mean(
                    event_error_gain
                    > NUMERIC_TOLERANCE
                )
            ),

            "correction_mean_averaging_gain_pp": float(
                merged[
                    "averaging_gain_pp_correction"
                ].mean()
            ),

            "combination_mean_averaging_gain_pp": float(
                merged[
                    "averaging_gain_pp_combination"
                ].mean()
            ),

            "combination_averaging_gain_difference_pp": float(
                averaging_gain_difference.mean()
            ),
        }
    )


correction_learning_stability = pd.DataFrame(
    comparison_rows
)


correction_learning_stability.to_csv(
    CORRECTION_LEARNING_STABILITY_PATH,
    index=False,
)


print("Saved:", CORRECTION_LEARNING_STABILITY_PATH)

display(correction_learning_stability)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_correction_vs_learning_stability.csv


,topology,prior_view,correction_experiment,combination_experiment,n_events,correction_mean_window_std_pp,combination_mean_window_std_pp,combination_std_reduction_pp,combination_more_stable_std_rate_pct,correction_mean_window_range_pp,combination_mean_window_range_pp,combination_range_reduction_pp,combination_more_stable_range_rate_pct,correction_event_mae_pp,combination_event_mae_pp,combination_event_mae_gain_pp,combination_lower_event_error_rate_pct,correction_mean_averaging_gain_pp,combination_mean_averaging_gain_pp,combination_averaging_gain_difference_pp
0,110kv,1E,C110-1E,L110-1E,912,0.478937,1.215045,-0.736109,17.214912,1.227068,3.110435,-1.883367,16.995614,6.636678,7.825728,-1.189049,28.728070,0.043890,0.142787,0.098897
1,110kv,2E,C110-2E,L110-2E,912,0.841788,0.564757,0.277031,64.364035,2.129670,1.406305,0.723365,64.473684,0.468317,0.238907,0.229410,61.184211,0.376387,0.259979,-0.116408


In [41]:
worst_event_frames = []


for experiment_id in EXPERIMENT_ORDER:
    experiment_frame = event_stability.loc[
        event_stability["experiment_id"]
        == experiment_id
    ].copy()

    most_unstable = (
        experiment_frame
        .sort_values(
            [
                "window_prediction_range_pp",
                "event_mean_abs_error_pp",
            ],
            ascending=[
                False,
                False,
            ],
            kind="stable",
        )
        .head(25)
        .copy()
    )

    most_unstable[
        "instability_rank"
    ] = np.arange(
        1,
        len(most_unstable) + 1,
    )

    worst_event_frames.append(
        most_unstable
    )


worst_unstable_events = pd.concat(
    worst_event_frames,
    ignore_index=True,
)


worst_unstable_events.to_csv(
    WORST_UNSTABLE_EVENTS_PATH,
    index=False,
)


print("Saved:", WORST_UNSTABLE_EVENTS_PATH)

display(
    worst_unstable_events[
        [
            "experiment_id",
            "instability_rank",
            "sample_id",
            "case",
            "y_fault_line",
            "y_true_pct",
            "event_mean_prediction_pct",
            "event_mean_abs_error_pp",
            "window_prediction_std_pp",
            "window_prediction_range_pp",
            "averaging_gain_pp",
        ]
    ].head(20)
)

Saved: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_worst_unstable_events.csv


,experiment_id,instability_rank,sample_id,case,y_fault_line,y_true_pct,event_mean_prediction_pct,event_mean_abs_error_pp,window_prediction_std_pp,window_prediction_range_pp,averaging_gain_pp
0,C110-1E,1,49,slg_c,MainLn1-2A,80.000001,70.311794,9.688208,5.329491,13.941455,0.000000
1,C110-1E,2,36,slg_b,MainLn2-3A,80.000001,84.795651,4.795650,5.236880,13.251787,0.139952
2,C110-1E,3,41,slg_b,MainLn2-3B,80.000001,91.479237,11.479236,5.567699,13.031620,0.000000
3,C110-1E,4,59,slg_c,MainLn2-3A,80.000001,90.436690,10.436688,5.610097,12.088072,0.000000
4,C110-1E,5,110,slg_b,MainLn2-3B,80.000001,95.143811,15.143810,4.291725,11.763543,0.000000
5,C110-1E,6,128,slg_c,MainLn2-3A,80.000001,73.813425,6.186576,4.355598,11.111712,0.000000
6,C110-1E,7,740,llg_ca,MainLn1-2A,99.000001,92.580546,6.419455,3.719464,9.207487,0.000000
7,C110-1E,8,625,llg_ab,MainLn1-2A,99.000001,96.035512,2.964489,3.114930,8.732527,0.500000
8,C110-1E,9,703,llg_ab,MainLn2-3A,80.000001,81.934799,1.934798,3.230032,8.393109,1.381975
9,C110-1E,10,64,slg_c,MainLn2-3B,80.000001,96.187733,16.187732,3.847394,8.358103,0.000000


In [42]:
print("=" * 130)
print("110 kV WINDOW-STABILITY SUMMARY")
print("=" * 130)

print(
    window_headline[
        [
            "experiment_id",
            "event_mean_mae_pp",
            "mean_window_mae_pp",
            "mean_averaging_gain_pp",
            "averaging_helped_rate_pct",
            "averaging_hurt_rate_pct",
            "mean_window_prediction_std_pp",
            "p95_window_prediction_range_pp",
            "best_individual_window_idx",
            "best_individual_window_mae_pp",
            "worst_individual_window_idx",
            "worst_individual_window_mae_pp",
            "spearman_range_vs_event_error",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)


print("\n")
print("=" * 130)
print("CORRECTION VERSUS COMBINATION-LEARNING STABILITY")
print("=" * 130)

print(
    correction_learning_stability.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)


print("\n")
print("=" * 130)
print("MOST UNSTABLE FAULT CASE PER EXPERIMENT")
print("=" * 130)

print(
    worst_case_stability[
        [
            "experiment_id",
            "subgroup_value",
            "support",
            "mean_window_prediction_std_pp",
            "mean_window_prediction_range_pp",
            "mean_averaging_gain_pp",
            "averaging_helped_rate_pct",
            "event_mean_mae_pp",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.6f}"
        ),
    )
)

110 kV WINDOW-STABILITY SUMMARY
experiment_id  event_mean_mae_pp  mean_window_mae_pp  mean_averaging_gain_pp  averaging_helped_rate_pct  averaging_hurt_rate_pct  mean_window_prediction_std_pp  p95_window_prediction_range_pp  best_individual_window_idx  best_individual_window_mae_pp  worst_individual_window_idx  worst_individual_window_mae_pp  spearman_range_vs_event_error
      C110-1E           6.636678            6.680568                0.043890                   9.429825                 0.000000                       0.478937                        4.842148                           8                       6.650962                           10                        6.717914                       0.134324
      L110-1E           7.825728            7.968515                0.142787                  15.460526                 0.000000                       1.215045                       11.905118                          11                       7.846939                            9   

In [43]:
all_experiments_complete = bool(
    set(
        window_headline[
            "experiment_id"
        ]
    )
    == set(EXPERIMENT_ORDER)
)


all_window_indices_complete = bool(
    len(window_index_metrics) == 16
    and set(
        window_index_metrics[
            "window_idx"
        ].astype(int)
    )
    == EXPECTED_WINDOWS
)


completion_summary = {
    "notebook": (
        "05_posthoc_110kv_window_stability"
    ),
    "status": "PASS",
    "analysis_unit": (
        "physical_event_with_four_retained_windows"
    ),
    "experiments": EXPERIMENT_ORDER,
    "event_stability_rows": int(
        len(event_stability)
    ),
    "window_index_metric_rows": int(
        len(window_index_metrics)
    ),
    "all_experiments_complete": (
        all_experiments_complete
    ),
    "all_window_indices_complete": (
        all_window_indices_complete
    ),
    "event_stability": str(
        EVENT_STABILITY_PATH
    ),
    "window_index_metrics": str(
        WINDOW_INDEX_METRICS_PATH
    ),
    "window_headline_metrics": str(
        WINDOW_HEADLINE_PATH
    ),
    "stability_subgroups": str(
        STABILITY_SUBGROUP_PATH
    ),
    "correction_vs_learning_stability": str(
        CORRECTION_LEARNING_STABILITY_PATH
    ),
    "worst_unstable_events": str(
        WORST_UNSTABLE_EVENTS_PATH
    ),
    "core_posthoc_analysis_complete": True,
    "next_step": (
        "draft_posthoc_section_and_generate_figures"
    ),
}


with COMPLETION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        completion_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("\n")
print("=" * 80)
print("NOTEBOOK 05 COMPLETE")
print("=" * 80)

print("Status: PASS")

print(
    "Experiments analyzed:",
    len(EXPERIMENT_ORDER),
)

print(
    "Event-stability rows:",
    len(event_stability),
)

print(
    "Window-index metric rows:",
    len(window_index_metrics),
)

print(
    "All experiments complete:",
    all_experiments_complete,
)

print(
    "All window indices complete:",
    all_window_indices_complete,
)

print(
    "\nWindow headline metrics:",
    WINDOW_HEADLINE_PATH,
)

print(
    "Correction-versus-learning stability:",
    CORRECTION_LEARNING_STABILITY_PATH,
)

print(
    "\nCORE POST-HOC ANALYSIS COMPLETE."
)

print(
    "The post-hoc section can now be written."
)



NOTEBOOK 05 COMPLETE
Status: PASS
Experiments analyzed: 4
Event-stability rows: 3648
Window-index metric rows: 16
All experiments complete: True
All window indices complete: True

Window headline metrics: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_window_headline_metrics.csv
Correction-versus-learning stability: /home/hpc/iwi5/iwi5305h/Masters_thesis_PR_LABS/outputs/chapter4/posthoc_final/posthoc_110kv_correction_vs_learning_stability.csv

CORE POST-HOC ANALYSIS COMPLETE.
The post-hoc section can now be written.
